# CBIS-DDSM Whole-Mammogram Malignancy Classifier

Single-view, whole-mammogram **benign vs malignant** classifier trained on CBIS-DDSM on one Colab A100.
Specification: [`docs/PRD.md`](../docs/PRD.md).

> ⚠️ **Not for clinical use.** Portfolio and research project only. CBIS-DDSM contains 1990s scanned film, no normal
> (lesion-free) images, and imprecise annotations. AUCs from this benign-vs-malignant task must never be compared with
> cancer-vs-normal screening AUCs.

| | |
|---|---|
| **Primary metric** | Image-level ROC AUC, 5-fold patient-grouped CV inside the official training split (mean ± SD and pooled out-of-fold). Target ≥ 0.90 (stretch; see PRD §3). |
| **Secondary metric** | Official test-split AUC with 95% patient-level bootstrap CI, computed **once** from a frozen, hashed model. |
| **Honesty rule** | If 0.90 is missed, the best score and the gap analysis are reported. Nothing is tuned on test. |

**Dataset citation (CC BY 3.0).** Sawyer-Lee R., Gimenez F., Hoogi A., Rubin D. *Curated Breast Imaging Subset of DDSM* [Data set].
The Cancer Imaging Archive, 2016. DOI: 10.7937/K9/TCIA.2016.7O02S9CY · Lee R.S. et al. *A curated mammography data set
for use in computer-aided detection and diagnosis research.* Scientific Data 4, 170177 (2017) · Clark K. et al.
*The Cancer Imaging Archive (TCIA).* J Digit Imaging 26, 1045–1057 (2013).

### Results
The table below is generated from `results/metrics.json` by code. It is empty until the final evaluation has run.

In [ ]:
# Results table — rendered from metrics.json only; numbers are never typed by hand.
import json, os
from pathlib import Path

_candidates = [Path("results/metrics.json"), Path("../results/metrics.json"),
               Path("/content/drive/MyDrive/cbis-ddsm-mammo-classifier/results/metrics.json")]
_found = next((p for p in _candidates if p.exists()), None)
if _found is None:
    print("No metrics.json yet — results appear here after milestones 13–14 have run.")
else:
    _m = json.loads(_found.read_text())
    _rows = [(k, v.get("split", "?"), v.get("value"), v.get("ci95")) for k, v in _m.get("metrics", {}).items()]
    print(f"Source: {_found}")
    print(f"{'metric':45s} {'split':28s} {'value':>8s}  95% CI")
    for k, split, val, ci in _rows:
        print(f"{k:45s} {split:28s} {val if val is None else f'{val:.4f}':>8}  {ci}")

## 1. Config

One dataclass holds every path, seed and flag. Expensive stages sit behind flags and skip themselves when their cached
outputs exist. **`FINAL_EVAL` stays `False`** until review gate 3 is approved (PRD §6, §12).

Review gates are flags too: a gate cell raises and stops *Run all* until its flag is set to `True` after review.

In [ ]:
import os, sys, re, json, time, math, random, shutil, hashlib, platform, subprocess
from dataclasses import dataclass, asdict, field
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


@dataclass
class Config:
    # ---- identity / reproducibility
    project: str = "cbis-ddsm-mammo-classifier"
    seed: int = 42

    # ---- paths (Drive = persistent cache; local = fast scratch on the Colab VM)
    drive_root: str = "/content/drive/MyDrive/cbis-ddsm-mammo-classifier"
    local_root: str = "/content/data"

    # ---- data acquisition
    data_route: str = "idc"            # "idc" (preferred) or "kaggle" (fallback, PRD §5.9)
    idc_collection_hint: str = "cbis"  # substring used to *look up* the IDC collection ID; the ID itself is not assumed
    download_dicom: bool = True        # False = metadata-only run (CSVs + IDC index, no image files)
    download_roi_masks: bool = False   # only needed for patch pre-training / Grad-CAM ROI outlines
    download_chunk_series: int = 250   # series per s5cmd call, so progress is visible and a restart loses little
    disk_headroom_gb: float = 25.0     # free space required beyond the download itself (PNG cache, checkpoints)
    min_free_ram_gb: float = 8.0

    # ---- preprocessing / cache (PRD §5.5–5.8)
    cache_version: str = "v1"          # bump when preprocessing changes, so an old cache is never reused silently
    cache_long_side: int = 2048        # stored PNG long side (PRD §5.7)
    cache_uncropped: bool = False      # also store an uncropped (oriented, resized) variant for the crop ablation
    otsu_long_side: int = 512          # breast segmentation runs on a copy downsampled to this long side
    border_trim_frac: float = 0.01     # ignore this fraction at each film edge when segmenting (scanner borders)
    open_kernel_frac: float = 0.03     # morphological opening (× otsu_long_side) breaks thin bridges to labels/borders
    crop_margin_frac: float = 0.02     # margin added around the breast bounding box
    n_workers: int = os.cpu_count() or 2
    delete_dicom_after_cache: bool = False  # free local disk once the train PNG cache is verified
    cv_seed: int = 42                  # StratifiedGroupKFold random_state (PRD §6)

    # ---- model / training defaults (used from milestone 9 onward)
    image_size: tuple = (1152, 896)    # (H, W)
    backbone: str = "efficientnet_b3"
    fold: int = 0
    n_folds: int = 5

    # ---- stage flags (PRD §10.1)
    RUN_DOWNLOAD: bool = True
    RUN_PREPROCESS: bool = True
    RUN_PATCH_PRETRAIN: bool = True
    RUN_CV: bool = True
    RUN_GRADCAM: bool = True         # milestone 15 (downloads only the ROI series of the images shown)
    n_error_examples: int = 8        # milestone 16: FP/FN examples shown per split
    hf_repo_id: str = ""             # milestone 17: optional Hugging Face Hub repo for the exported weights
    RUN_S0_DIAGNOSTIC: bool = True   # pipeline sanity checks after Stage 0 (dev data only; skips once done)
    FINAL_EVAL: bool = True            # approved in chat 2026-11-12 with gate 3 (the official test split is evaluated once)

    # ---- review gates (PRD §12): set True only after the human has reviewed the gate output
    GATE1_COUNTS_APPROVED: bool = True   # approved 2026-11-10 (see the gate 1 decision below milestone 5)
    # Gate 1 approval holds only for exactly these reviewed mismatches; any new or different one re-closes the gate.
    GATE1_REVIEWED_MISMATCHES: tuple = (
        "Full-mammogram images, total",
        "Full-mammogram images, official train (train+val)",
        "Full-mammogram images, official test",
        "Referenced full-image series with malformed IDC PatientID",
    )
    GATE2_BASELINE_APPROVED: bool = True   # approved in chat 2026-11-11 on Stage 0b (fold-0 AUC 0.6491 [0.5937, 0.7025])
    GATE3_PRE_TEST_APPROVED: bool = True   # approved in chat 2026-11-12 on the 5-fold CV of s3_tta_hflip
    # Training (batch probe, any model fit) only runs on an A100 unless this is set; data stages run on any runtime,
    # so a non-A100 session can download, cache and fold the data, then stop before Stage 0.
    allow_non_a100_training: bool = False

    # ---- derived paths
    @property
    def drive(self): return Path(self.drive_root)
    @property
    def local(self): return Path(self.local_root)
    @property
    def csv_dir(self): return self.drive / "metadata" / "csv"
    @property
    def meta_dir(self): return self.drive / "metadata"
    @property
    def results_dir(self): return self.drive / "results"
    @property
    def figures_dir(self): return self.results_dir / "figures"
    @property
    def ckpt_dir(self): return self.drive / "checkpoints"
    @property
    def cache_dir(self): return self.drive / "cache"
    @property
    def dicom_dir(self): return self.local / "dicom"
    @property
    def png_dir(self): return self.local / "png"

    def cache_tar(self, split): return self.cache_dir / f"png_{split}_{self.cache_version}.tar"
    def cache_marker(self, split): return self.cache_dir / f"png_{split}_{self.cache_version}.complete.json"
    def manifest_path(self, split): return self.meta_dir / f"manifest_{split}_{self.cache_version}.csv"


cfg = Config()

# Outside Colab (e.g. a local metadata-only dry run) paths can be redirected with environment variables.
if not IN_COLAB:
    cfg.drive_root = os.environ.get("CBIS_DRIVE_ROOT", str(Path.cwd() / "_drive"))
    cfg.local_root = os.environ.get("CBIS_LOCAL_ROOT", str(Path.cwd() / "_local"))

assert cfg.FINAL_EVAL is False or cfg.GATE3_PRE_TEST_APPROVED, "FINAL_EVAL requires review gate 3 approval"


def seed_everything(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


class ReviewGate(Exception):
    '''Raised to stop Run-all at a review gate until the gate flag is set after human review.'''


def review_gate(name: str, approved: bool, message: str):
    if approved:
        print(f"✅ {name}: approved — continuing.")
        return
    raise ReviewGate(f"⛔ {name} — stopping here for review.\n{message}\n"
                     f"After review, set the corresponding GATE flag in Config to True and re-run.")


seed_everything(cfg.seed)
print("IN_COLAB:", IN_COLAB)
print(json.dumps(asdict(cfg), indent=2, default=str))

## 2. Environment check

GPU name and memory, CUDA and torch versions, disk and RAM. On a non-A100 GPU the data stages run and training stops before Stage 0 (`allow_non_a100_training`).
Missing lightweight packages are installed; torch is whatever the Colab image ships (versions are pinned into
`requirements.txt` from this cell's output).

In [ ]:
import importlib


def sh(cmd: str, check: bool = False) -> str:
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}\n{out}")
    return out


REQUIRED = {"idc_index": "idc-index", "pydicom": "pydicom", "pandas": "pandas", "numpy": "numpy",
            "psutil": "psutil", "requests": "requests", "cv2": "opencv-python-headless",
            "sklearn": "scikit-learn", "matplotlib": "matplotlib"}
_missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print("Installing:", _missing)
    print(sh(f"{sys.executable} -m pip install -q " + " ".join(_missing), check=True)[-500:])

import numpy as np
import pandas as pd
import psutil

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

print(f"Python   {platform.python_version()}  ({platform.platform()})")
_versions = {}
for mod in ["torch", "torchvision", "timm", "albumentations", "sklearn", "pydicom", "idc_index", "pandas", "numpy"]:
    try:
        m = importlib.import_module(mod)
        _versions[mod] = getattr(m, "__version__", "?")
    except ImportError:
        _versions[mod] = None
print("Packages:", json.dumps(_versions))

# ---- GPU
print("\n--- nvidia-smi ---")
print(sh("nvidia-smi") or "nvidia-smi not available")
GPU_INFO = {"name": None, "total_gb": None}
try:
    import torch
    print(f"\ntorch {torch.__version__} | CUDA build {torch.version.cuda} | cuDNN {torch.backends.cudnn.version()}")
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        free_b, total_b = torch.cuda.mem_get_info(0)
        GPU_INFO = {"name": p.name, "total_gb": round(p.total_memory / 1e9, 1), "free_gb": round(free_b / 1e9, 1),
                    "capability": f"{p.major}.{p.minor}", "sm_count": p.multi_processor_count}
        print("GPU:", GPU_INFO)
        if "A100" not in p.name:
            print(f"⚠️  GPU is {p.name}, not an A100. Data stages run; training stops before Stage 0 unless cfg.allow_non_a100_training.")
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    else:
        print("⚠️  No CUDA GPU. Milestones 0–8 (data) run on CPU; training (≥ 12) needs a GPU runtime.")
except ImportError:
    print("⚠️  torch not installed (fine for the data milestones; Colab ships it).")

# ---- Disk and RAM
_disk_path = "/content" if Path("/content").exists() else str(Path(cfg.local_root).parent if Path(cfg.local_root).parent.exists() else Path.cwd())
print(f"\n--- df -h {_disk_path} ---")
print(sh(f"df -h {_disk_path}"))
_du = shutil.disk_usage(_disk_path)
_vm = psutil.virtual_memory()
RESOURCES = {"disk_path": _disk_path, "disk_free_gb": round(_du.free / 1e9, 1), "disk_total_gb": round(_du.total / 1e9, 1),
             "ram_total_gb": round(_vm.total / 1e9, 1), "ram_available_gb": round(_vm.available / 1e9, 1),
             "cpu_count": os.cpu_count()}
print("Resources:", RESOURCES)
if RESOURCES["ram_available_gb"] < cfg.min_free_ram_gb:
    print(f"⚠️  Only {RESOURCES['ram_available_gb']} GB RAM free (< {cfg.min_free_ram_gb}); RAM tensor caching will be disabled.")
# The hard disk-space stop happens in milestone 4, once the real download size is known from the IDC index.

## 3. Drive mount and folder layout

Drive holds everything that must survive a disconnect (CSVs, inventories, PNG cache tarball, checkpoints, results).
The Colab local disk holds the DICOMs and the untarred PNG cache for fast reads.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    # The Drive consent popup must be completed before the timeout; the default (2 min) is too short when the popup
    # appears mid Run-all and the account chooser + permission screen take a while. Wait up to 15 min instead.
    print("👉 Approve the 'Permit this notebook to access your Google Drive files?' prompt (tick every permission).",
          flush=True)
    drive.mount("/content/drive", timeout_ms=15 * 60 * 1000)

for d in [cfg.csv_dir, cfg.meta_dir, cfg.results_dir, cfg.figures_dir, cfg.ckpt_dir, cfg.cache_dir,
          cfg.dicom_dir, cfg.png_dir]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("Drive root :", cfg.drive)
print("Local root :", cfg.local)
for d in sorted({cfg.drive, cfg.local}):
    for p in sorted(Path(d).iterdir()):
        print("  ", p)

## 4. Download: metadata CSVs and full-mammogram DICOM via IDC

Idempotent: files already on disk are skipped.

1. The four case-description CSVs from the TCIA collection page (cached on Drive).
2. The IDC collection ID is **looked up** with `idc-index`, not assumed. Its `SeriesDescription` values are printed.
3. Full-mammogram series are selected **by the SeriesInstanceUIDs referenced in the CSVs' `image file path` column**,
   so no series description is hard-coded. Every CSV row must match an IDC series (100%, asserted).
4. Dry run prints the download size; the notebook stops if the disk is too small.
5. Download in chunks with `s5cmd sync` (resume-safe), then every CSV row is mapped to exactly one `.dcm` on disk.

Train DICOMs are downloaded only while the train PNG cache (milestone 7) is incomplete; a fresh runtime with a finished
cache downloads nothing. **Test DICOMs are downloaded only when `FINAL_EVAL` is `True`** — before that, the test split
is checked at metadata level only (UID match against the IDC index, which covers all 3,568 rows).

In [ ]:
# ---- 4.1 Metadata CSVs
import requests

TCIA_CSV_BASE = "https://www.cancerimagingarchive.net/wp-content/uploads/"
CSV_FILES = {
    ("mass", "train"): "mass_case_description_train_set.csv",
    ("calc", "train"): "calc_case_description_train_set.csv",
    ("mass", "test"): "mass_case_description_test_set.csv",
    ("calc", "test"): "calc_case_description_test_set.csv",
}


def fetch_csv(fname: str) -> Path:
    dst = cfg.csv_dir / fname
    if dst.exists() and dst.stat().st_size > 0:
        return dst
    url = TCIA_CSV_BASE + fname
    for attempt in range(1, 5):
        try:
            r = requests.get(url, timeout=120)
            r.raise_for_status()
            break
        except Exception as e:
            if attempt == 4:
                raise RuntimeError(f"Could not download {url} after {attempt} attempts ({e}).\nDownload it manually from "
                                   f"https://www.cancerimagingarchive.net/collection/cbis-ddsm/ and place it at {dst}") from e
            print(f"  {fname}: attempt {attempt} failed ({type(e).__name__}); retrying in {10 * attempt} s")
            time.sleep(10 * attempt)
    head = r.content[:200].decode("utf-8", errors="replace")
    assert "patient_id" in head, f"{url} did not return the expected CSV (starts with: {head[:120]!r})"
    tmp = dst.with_suffix(".part")
    tmp.write_bytes(r.content)
    tmp.rename(dst)
    return dst


CSV_PATHS = {k: fetch_csv(f) for k, f in CSV_FILES.items()}
RAW = {k: pd.read_csv(p, dtype=str) for k, p in CSV_PATHS.items()}
print(f"{'file':42s} {'rows':>6s} {'sha256[:12]':>14s}")
for k, p in CSV_PATHS.items():
    print(f"{p.name:42s} {len(RAW[k]):6d} {hashlib.sha256(p.read_bytes()).hexdigest()[:12]:>14s}")

In [ ]:
# ---- 4.2 Find the IDC collection and inspect its series
DATA_ROUTE = {"route": cfg.data_route}
IDC = None
if cfg.data_route == "idc":
    import idc_index
    from idc_index import IDCClient

    idc_client = IDCClient()
    _collections = idc_client.get_collections()
    _cands = [c for c in _collections if cfg.idc_collection_hint in c.lower()]
    assert len(_cands) == 1, f"Expected exactly one IDC collection matching {cfg.idc_collection_hint!r}, got {_cands}"
    IDC_COLLECTION_ID = _cands[0]
    try:
        _idc_version = idc_client.get_idc_version()
    except Exception:
        _idc_version = None
    DATA_ROUTE.update({"idc_collection_id": IDC_COLLECTION_ID, "idc_data_version": _idc_version,
                       "idc_index_version": idc_index.__version__})
    print("IDC collection ID:", IDC_COLLECTION_ID, "| IDC data version:", _idc_version,
          "| idc-index", idc_index.__version__)

    _need = {"collection_id", "PatientID", "StudyInstanceUID", "SeriesInstanceUID", "SeriesDescription",
             "instanceCount", "series_size_MB"}
    _have = set(idc_client.index.columns)
    assert _need <= _have, f"IDC index is missing columns {_need - _have}; available: {sorted(_have)}"
    IDC = idc_client.index[idc_client.index["collection_id"] == IDC_COLLECTION_ID].copy()

    print(f"\nCollection: {len(IDC):,} series | {int(IDC.instanceCount.sum()):,} instances | "
          f"{IDC.PatientID.nunique():,} PatientIDs | {IDC.series_size_MB.sum() / 1000:.1f} GB")
    print("\nSeriesDescription values (as found, not assumed):")
    print(IDC.groupby("SeriesDescription", dropna=False)
             .agg(series=("SeriesInstanceUID", "size"), instances=("instanceCount", "sum"),
                  size_GB=("series_size_MB", lambda s: round(s.sum() / 1000, 1))))

In [ ]:
# ---- 4.3 Map every CSV row to an IDC series through the UIDs in its image path
UID_RE = re.compile(r"^\d+(\.\d+)+$")


def parse_dicom_path(p: str):
    '''CSV path layout: <DICOM PatientID>/<StudyInstanceUID>/<SeriesInstanceUID>/<file>.dcm'''
    parts = str(p).strip().replace("\\", "/").split("/")
    if len(parts) < 4 or not UID_RE.match(parts[1]) or not UID_RE.match(parts[2]):
        return None, None, None
    return parts[0], parts[1], parts[2]


_rows = []
for (src, split), df in RAW.items():
    col = next(c for c in df.columns if c.strip().lower() == "image file path")
    for i, p in enumerate(df[col]):
        pid, study, series = parse_dicom_path(p)
        _rows.append({"source": src, "split": split, "row": i, "dicom_patient_id": pid,
                      "study_uid": study, "series_uid": series, "raw_path": str(p).strip()})
ROW_SERIES = pd.DataFrame(_rows)
_bad = ROW_SERIES[ROW_SERIES.series_uid.isna()]
assert _bad.empty, f"{len(_bad)} CSV image paths could not be parsed:\n{_bad.head(20)}"

if IDC is not None:
    _idc_s = IDC.set_index("SeriesInstanceUID")
    ROW_SERIES["in_idc"] = ROW_SERIES.series_uid.isin(_idc_s.index)
    _un = ROW_SERIES[~ROW_SERIES.in_idc]
    if not _un.empty:
        _un = _un.assign(patient_in_idc=_un.dicom_patient_id.isin(set(IDC.PatientID)),
                         study_in_idc=_un.study_uid.isin(set(IDC.StudyInstanceUID)))
        raise AssertionError(f"{len(_un)} of {len(ROW_SERIES)} CSV rows have no matching IDC series. "
                             f"Offending rows:\n{_un.to_string(max_rows=50)}")
    print(f"✅ 100% of {len(ROW_SERIES):,} CSV rows match an IDC SeriesInstanceUID.")
    m = _idc_s.loc[ROW_SERIES.series_uid]
    ROW_SERIES["idc_patient_id"] = m["PatientID"].values
    ROW_SERIES["idc_series_description"] = m["SeriesDescription"].values
    _std_ok = m["StudyInstanceUID"].values == ROW_SERIES.study_uid.values
    assert _std_ok.all(), f"StudyInstanceUID disagrees with IDC for {(~_std_ok).sum()} rows"
    print("✅ StudyInstanceUID in the CSV paths agrees with IDC for every row.")

    # PatientID check. Known source defect (found when this notebook was first run, IDC v25): some Calc-Test
    # full-image series carry the DICOM PatientID '<P_xxxxx>_<SIDE>_<VIEW>.dcm' instead of
    # 'Calc-Test_<P_xxxxx>_<SIDE>_<VIEW>', and an empty SeriesDescription. Only that exact pattern is accepted
    # (the series and study UIDs still match); any other disagreement stops the notebook.
    _pid_bad = ROW_SERIES[ROW_SERIES.idc_patient_id != ROW_SERIES.dicom_patient_id].copy()
    _known = (_pid_bad.dicom_patient_id.str.startswith("Calc-Test_")
              & (_pid_bad.idc_patient_id == _pid_bad.dicom_patient_id.str.replace("Calc-Test_", "", regex=False) + ".dcm"))
    assert _known.all(), ("PatientID disagrees with IDC in an unexpected way for "
                          f"{(~_known).sum()} rows:\n{_pid_bad[~_known].head(30).to_string()}")
    IDC_PATIENTID_ANOMALY = {"rows": int(len(_pid_bad)), "series": int(_pid_bad.series_uid.nunique()),
                             "csv_files": sorted(set(_pid_bad.source + "/" + _pid_bad.split))}
    if len(_pid_bad):
        print(f"⚠️  Known IDC metadata defect: {IDC_PATIENTID_ANOMALY['series']} full-image series "
              f"({IDC_PATIENTID_ANOMALY['rows']} CSV rows, {IDC_PATIENTID_ANOMALY['csv_files']}) have DICOM PatientID "
              f"'P_xxxxx_SIDE_VIEW.dcm' instead of 'Calc-Test_P_xxxxx_SIDE_VIEW'. Matched by Series/Study UID; "
              "split and patient are taken from the CSV, never from the DICOM PatientID.")
    else:
        print("✅ PatientID in the CSV paths agrees with IDC for every row.")

    # Split comes from which CSV referenced the series — NOT from the DICOM PatientID (see defect above).
    _series_split = ROW_SERIES.groupby("series_uid").split.agg(lambda s: "|".join(sorted(set(s))))
    assert _series_split.isin(["train", "test"]).all(), "A full-image series is referenced by both train and test CSVs"
    FULL_SERIES = IDC[IDC.SeriesInstanceUID.isin(_series_split.index)].copy()
    FULL_SERIES["split"] = FULL_SERIES.SeriesInstanceUID.map(_series_split)
    print("\nSeriesDescription of the series the CSVs reference as full images:")
    print(FULL_SERIES.SeriesDescription.value_counts(dropna=False).to_string())
    assert (FULL_SERIES.instanceCount == 1).all(), "Some referenced full-image series have != 1 instance"
    print(f"Referenced full-image series: {len(FULL_SERIES):,} "
          f"(train {int((FULL_SERIES.split == 'train').sum()):,}, test {int((FULL_SERIES.split == 'test').sum()):,})")

In [ ]:
# ---- 4.4 Dry run, disk check, chunked resumable download, inventory
#
# Which splits need DICOM *this session*:
#   train — only if its PNG cache (milestone 7) is not complete on Drive yet;
#   test  — only when FINAL_EVAL is True (test files are not even downloaded before review gate 3).
# So a fresh runtime with a finished cache re-downloads nothing ("Run all with default flags using cached data").


def series_on_disk(series_uid: str):
    d = cfg.dicom_dir / series_uid
    return sorted(d.glob("*.dcm")) if d.is_dir() else []


def cache_complete(split: str) -> bool:
    return cfg.cache_marker(split).exists()


INVENTORY_PATH = cfg.meta_dir / "dicom_inventory.csv"
SPLITS_NEEDING_DICOM = [s for s in ["train"] + (["test"] if cfg.FINAL_EVAL else []) if not cache_complete(s)]
print("PNG cache complete on Drive:", {s: cache_complete(s) for s in ["train", "test"]})
print("Splits needing DICOM this session:", SPLITS_NEEDING_DICOM or "none")

INVENTORY = pd.read_csv(INVENTORY_PATH) if INVENTORY_PATH.exists() else None
DOWNLOADED_THIS_SESSION = False

if cfg.data_route == "idc" and cfg.download_dicom and cfg.RUN_DOWNLOAD and SPLITS_NEEDING_DICOM:
    sel = FULL_SERIES[FULL_SERIES.split.isin(SPLITS_NEEDING_DICOM)]
    todo = [u for u in sel.SeriesInstanceUID if len(series_on_disk(u)) == 0]
    need_gb = sel[sel.SeriesInstanceUID.isin(todo)].series_size_MB.sum() / 1000
    print(f"Selected {len(sel):,} series ({sel.series_size_MB.sum() / 1000:.1f} GB); "
          f"{len(sel) - len(todo):,} already on disk; {len(todo):,} to fetch ({need_gb:.1f} GB).")

    if todo:
        idc_client.download_from_selection(downloadDir=str(cfg.dicom_dir), seriesInstanceUID=todo, dry_run=True)
        print(f"Dry run OK: {len(todo):,} series, {need_gb:.1f} GB to download (size from the IDC index).")
        free_gb = shutil.disk_usage(cfg.dicom_dir).free / 1e9
        if free_gb < need_gb + cfg.disk_headroom_gb:
            raise RuntimeError(
                f"Not enough disk: {free_gb:.1f} GB free, need {need_gb:.1f} GB + {cfg.disk_headroom_gb} GB headroom "
                "(PNG cache, tarball). Use a runtime with a larger disk.")

        t0 = time.time()
        n = cfg.download_chunk_series
        for i in range(0, len(todo), n):
            chunk = todo[i:i + n]
            idc_client.download_from_selection(downloadDir=str(cfg.dicom_dir), seriesInstanceUID=chunk,
                                               dirTemplate="%SeriesInstanceUID", use_s5cmd_sync=True,
                                               show_progress_bar=False, quiet=True)
            done = i + len(chunk)
            el = time.time() - t0
            print(f"  {done:5d}/{len(todo)} series  {el / 60:5.1f} min  ETA {el / done * (len(todo) - done) / 60:5.1f} min",
                  flush=True)

    # Inventory: exactly one .dcm per selected series; merged with any earlier inventory (e.g. train, then test)
    inv = []
    for u in sel.SeriesInstanceUID:
        files = series_on_disk(u)
        inv.append({"series_uid": u, "split": sel.set_index("SeriesInstanceUID").at[u, "split"], "n_files": len(files),
                    "dicom_path": str(files[0]) if files else None, "bytes": files[0].stat().st_size if files else 0})
    new_inv = pd.DataFrame(inv)
    bad = new_inv[new_inv.n_files != 1]
    assert bad.empty, f"{len(bad)} series do not have exactly one .dcm on disk:\n{bad.head(30).to_string()}"
    INVENTORY = (pd.concat([INVENTORY, new_inv]) if INVENTORY is not None else new_inv) \
        .drop_duplicates("series_uid", keep="last").reset_index(drop=True)
    INVENTORY.to_csv(INVENTORY_PATH, index=False)
    DOWNLOADED_THIS_SESSION = True
    print(f"✅ Inventory: {len(new_inv):,} series this session, one .dcm each, "
          f"{new_inv.bytes.sum() / 1e9:.1f} GB on disk → {INVENTORY_PATH}")

    # Every CSV row of the selected splits must resolve to a file on disk
    _rows_needed = ROW_SERIES[ROW_SERIES.split.isin(SPLITS_NEEDING_DICOM)]
    _miss = _rows_needed[~_rows_needed.series_uid.isin(set(new_inv.series_uid))]
    assert _miss.empty, f"{len(_miss)} CSV rows have no DICOM on disk:\n{_miss.head(30).to_string()}"
    print(f"✅ 100% of {len(_rows_needed):,} CSV rows ({'+'.join(SPLITS_NEEDING_DICOM)}) map to a DICOM file on disk.")

    # Header spot-check on TRAIN files only (test headers stay unread until FINAL_EVAL)
    import pydicom
    _pool = new_inv[new_inv.split == "train"]
    if len(_pool):
        _sample = _pool.sample(min(25, len(_pool)), random_state=cfg.seed)
        _hdr = []
        for _, r in _sample.iterrows():
            ds = pydicom.dcmread(r.dicom_path, stop_before_pixels=True)
            assert ds.SeriesInstanceUID == r.series_uid, f"UID mismatch in {r.dicom_path}"
            _hdr.append({"Photometric": ds.get("PhotometricInterpretation"), "BitsStored": ds.get("BitsStored"),
                         "Rows": ds.get("Rows"), "Columns": ds.get("Columns")})
        print(f"Header spot-check ({len(_sample)} train files, SeriesInstanceUID verified):")
        print(pd.DataFrame(_hdr).value_counts(["Photometric", "BitsStored"]).to_string())
elif not SPLITS_NEEDING_DICOM:
    print("✅ Nothing to download: the PNG cache covers every split needed this session.")
else:
    print("ℹ️  DICOM download skipped (download_dicom/RUN_DOWNLOAD False). "
          + ("Using the cached inventory." if INVENTORY is not None else
             "No cached inventory: metadata-only run; file-match checks are reported as not run."))

In [ ]:
# ---- 4.5 Kaggle JPEG mirror fallback (PRD §5.9) — inspection only; never used silently.
if cfg.data_route == "kaggle":
    from google.colab import userdata  # credentials come from Colab Secrets, never from notebook text
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    if importlib.util.find_spec("kagglehub") is None:
        print(sh(f"{sys.executable} -m pip install -q kagglehub", check=True))
    import kagglehub
    from PIL import Image

    kpath = Path(kagglehub.dataset_download("awsaf49/cbis-ddsm-breast-cancer-image-dataset"))
    exts = pd.Series([p.suffix.lower() for p in kpath.rglob("*") if p.is_file()]).value_counts()
    print("Kaggle mirror at", kpath, "\nFile types:\n", exts.to_string())
    for p in list(kpath.rglob("*.jpg"))[:3]:
        im = Image.open(p)
        a = np.asarray(im)
        print(f"{p.relative_to(kpath)}: mode={im.mode} size={im.size} dtype={a.dtype} min={a.min()} max={a.max()}")
    DATA_ROUTE.update({"kaggle_path": str(kpath), "kaggle_file_types": exts.to_dict()})
    (cfg.results_dir / "data_route.json").write_text(json.dumps(DATA_ROUTE, indent=2))
    raise NotImplementedError("Kaggle route inspected (see above). The CSV→file mapping for this layout must be "
                              "reviewed and written before the mirror is used; stopping rather than guessing.")

(cfg.results_dir / "data_route.json").write_text(json.dumps(DATA_ROUTE, indent=2, default=str))
print("Data route recorded:", DATA_ROUTE)

## 5. Metadata build

* Column names are **read from the CSVs** and normalised (lower-case, non-alphanumerics → `_`); the mapping is printed.
  Required columns are asserted.
* One row per **image** = (official split, base patient ID, breast side, view). Mass and calcification rows for the same
  image are merged; the number merged is logged.
* **Label** = 1 if any abnormality on the image is `MALIGNANT`, else 0 (`BENIGN` and `BENIGN_WITHOUT_CALLBACK`).
* **Grouping key** = base patient ID (`P_00038`), with Mass/Calc, Training/Test prefixes and side/view suffixes removed.
* A **counts table** re-checks every PRD §2 fact in code. Mismatches are reported, never patched. → **Review gate 1.**

Only metadata is read here; no image pixels from any split.

In [ ]:
# ---- 5.1 Normalise and validate columns
def norm_col(c: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", c.strip().lower()).strip("_")


REQUIRED_COLS = ["patient_id", "breast_density", "left_or_right_breast", "image_view", "abnormality_id",
                 "abnormality_type", "assessment", "pathology", "subtlety", "image_file_path",
                 "cropped_image_file_path", "roi_mask_file_path"]
EXPECTED_PREFIX = {("mass", "train"): "Mass-Training", ("calc", "train"): "Calc-Training",
                   ("mass", "test"): "Mass-Test", ("calc", "test"): "Calc-Test"}

frames = []
for (src, split), df in RAW.items():
    mapping = {c: norm_col(c) for c in df.columns}
    changed = {k: v for k, v in mapping.items() if k != v}
    print(f"{src}/{split}: renamed {changed}")
    d = df.rename(columns=mapping)
    missing = [c for c in REQUIRED_COLS if c not in d.columns]
    assert not missing, f"{src}/{split} is missing columns {missing}; has {list(d.columns)}"
    d = d.assign(source=src, split=split)
    frames.append(d)
ABN = pd.concat(frames, ignore_index=True, sort=False)
for c in ABN.columns:
    if ABN[c].dtype == object:
        ABN[c] = ABN[c].str.strip()

# ---- 5.2 Parse keys and validate values
ABN["pathology"] = ABN.pathology.str.upper()
ABN["side"] = ABN.left_or_right_breast.str.upper()
ABN["view"] = ABN.image_view.str.upper()
_p = ABN.image_file_path.map(parse_dicom_path)
ABN["dicom_patient_id"] = _p.str[0]
ABN["series_uid"] = _p.str[2]
ABN["base_pid"] = ABN.patient_id.str.extract(r"(P_\d{5})", expand=False)

assert ABN.base_pid.notna().all(), f"Unparseable patient_id: {ABN.loc[ABN.base_pid.isna(), 'patient_id'].unique()[:20]}"
assert set(ABN.side) <= {"LEFT", "RIGHT"}, f"Unexpected side values: {set(ABN.side)}"
assert set(ABN.view) <= {"CC", "MLO"}, f"Unexpected view values: {set(ABN.view)}"
PATHOLOGY_CLASSES = {"MALIGNANT", "BENIGN", "BENIGN_WITHOUT_CALLBACK"}
assert set(ABN.pathology) <= PATHOLOGY_CLASSES, f"Unexpected pathology values: {set(ABN.pathology) - PATHOLOGY_CLASSES}"

# Row values are read with r["col"], not r.col: on pandas 2.x `r.view` is the Series.view *method*, not the column.
_expected_pid = (ABN.apply(lambda r: f"{EXPECTED_PREFIX[(r['source'], r['split'])]}_{r['base_pid']}_{r['side']}_{r['view']}",
                           axis=1))
_bad = ABN[_expected_pid != ABN.dicom_patient_id]
assert _bad.empty, ("Image path PatientID disagrees with patient_id/side/view/prefix for "
                    f"{len(_bad)} rows:\n{_bad[['source', 'split', 'patient_id', 'side', 'view', 'dicom_patient_id']].head(20)}")
print(f"✅ {len(ABN):,} abnormality rows: columns, pathology classes, side/view values and path prefixes all validated.")

# Within one CSV, all abnormalities of an image must point to the same full-image series
_per = ABN.groupby(["source", "split", "base_pid", "side", "view"]).series_uid.nunique()
assert (_per == 1).all(), f"{(_per > 1).sum()} images point to >1 full-image series within one CSV:\n{_per[_per > 1].head(20)}"

In [ ]:
# ---- 5.3 Collapse abnormality rows to one row per image
KEY = ["split", "base_pid", "side", "view"]


def _first_sorted_unique(s):
    u = sorted(set(s.dropna()))
    return u[0] if u else None


def collapse(g: pd.DataFrame) -> pd.Series:
    srcs = set(g.source)
    dens = sorted(set(g.breast_density.dropna()))
    mass_series = sorted(set(g.loc[g.source == "mass", "series_uid"]))
    calc_series = sorted(set(g.loc[g.source == "calc", "series_uid"]))
    return pd.Series({
        "label": int((g.pathology == "MALIGNANT").any()),
        "abn_type": "both" if len(srcs) == 2 else ("mass" if "mass" in srcs else "calc"),
        "n_abnormalities": len(g),
        "n_mass": int((g.source == "mass").sum()),
        "n_calc": int((g.source == "calc").sum()),
        "pathologies": "|".join(sorted(set(g.pathology))),
        "breast_density": dens[-1] if dens else None,  # max if the CSVs disagree (flagged below)
        "density_conflict": len(dens) > 1,
        "assessment_max": pd.to_numeric(g.assessment, errors="coerce").max(),
        "subtlety_max": pd.to_numeric(g.subtlety, errors="coerce").max(),
        # canonical file: the mass-CSV series if present, else the calc one (deterministic, documented rule)
        "series_uid": (mass_series or calc_series)[0],
        "all_series_uids": "|".join(mass_series + calc_series),
        "n_series": len(set(mass_series + calc_series)),
        "dicom_patient_id": _first_sorted_unique(g.loc[g.series_uid == (mass_series or calc_series)[0], "dicom_patient_id"]),
    })


IMAGES = ABN.groupby(KEY, sort=True).apply(collapse, include_groups=False).reset_index()
IMAGES["image_key"] = IMAGES.split + "_" + IMAGES.base_pid + "_" + IMAGES.side + "_" + IMAGES.view
assert IMAGES.image_key.is_unique

n_merged_multi_abn = int((IMAGES.n_abnormalities > 1).sum())
n_merged_mass_calc = int((IMAGES.abn_type == "both").sum())
n_multi_series = int((IMAGES.n_series > 1).sum())
print(f"{len(ABN):,} abnormality rows → {len(IMAGES):,} images")
print(f"  images merged from >1 abnormality row      : {n_merged_multi_abn:,}")
print(f"  images present in both mass and calc CSVs  : {n_merged_mass_calc:,} "
      f"(of which {n_multi_series:,} have two distinct DICOM series; canonical = mass series)")
print(f"  images with conflicting breast density     : {int(IMAGES.density_conflict.sum()):,}")

if INVENTORY is not None:
    IMAGES = IMAGES.merge(INVENTORY[["series_uid", "dicom_path"]], on="series_uid", how="left")
    for _sp in set(INVENTORY.split):
        _miss = IMAGES[(IMAGES.split == _sp) & IMAGES.dicom_path.isna()]
        assert _miss.empty, f"{len(_miss)} {_sp} images are missing from the DICOM inventory:\n{_miss.head(20)}"
else:
    IMAGES["dicom_path"] = None

# ---- Cross-split patient overlap (reported here; enforced in milestone 8)
_train_p = set(IMAGES.loc[IMAGES.split == "train", "base_pid"])
_test_p = set(IMAGES.loc[IMAGES.split == "test", "base_pid"])
CROSS_SPLIT_PIDS = sorted(_train_p & _test_p)
print(f"\nBase patients: train {len(_train_p):,} | test {len(_test_p):,} | in both {len(CROSS_SPLIT_PIDS):,}"
      f" (these will be dropped from train/val in milestone 8, never from test)")

IMAGES.to_csv(cfg.meta_dir / "images.csv", index=False)
ABN.to_csv(cfg.meta_dir / "abnormalities.csv", index=False)
print(f"Saved {cfg.meta_dir / 'images.csv'} and abnormalities.csv")

# ---- Descriptive tables (not checks)
print("\nImages per split × label (0 = benign, 1 = malignant):")
print(pd.crosstab(IMAGES.split, IMAGES.label, margins=True).to_string())
print("\nImages per split × abnormality type:")
print(pd.crosstab(IMAGES.split, IMAGES.abn_type, margins=True).to_string())
print("\nMalignant prevalence by split × abnormality type:")
print(IMAGES.pivot_table(index="split", columns="abn_type", values="label", aggfunc="mean").round(3).to_string())

In [ ]:
# ---- 5.4 Counts table: every PRD §2 fact re-checked in code (mismatches are reported, never patched)
checks = []


def check(fact, expected, observed, source, how, abs_tol=0.0, rel_tol=0.0):
    '''Exact match unless a tolerance is given (only for facts the PRD states approximately).'''
    if observed is None:
        ok, exp_s, obs_s = None, expected, "not run"
    elif isinstance(expected, (set, frozenset)):
        ok = set(observed) == set(expected)
        exp_s, obs_s = sorted(expected), sorted(observed)
    else:
        ok = abs(observed - expected) <= max(abs_tol, rel_tol * abs(expected))
        exp_s, obs_s = expected, observed
    checks.append({"fact": fact, "expected": exp_s, "observed": obs_s,
                   "match": "n/a" if ok is None else ("✅" if ok else "❌"), "PRD source": source, "how computed": how})


n_img = lambda split=None: int(len(IMAGES) if split is None else (IMAGES.split == split).sum())

check("Full-mammogram images, total", 3103, n_img(), "Petrini 2025 Tab. 1",
      "unique (split, base_pid, side, view) over the 4 CSVs")
check("Full-mammogram images, official train (train+val)", 2212 + 246, n_img("train"), "Petrini 2025 Tab. 1",
      "as above, train CSVs")
check("Full-mammogram images, official test", 645, n_img("test"), "Petrini 2025 Tab. 1", "as above, test CSVs")
check("Real participants", 1566, int(IMAGES.base_pid.nunique()), "TCIA",
      "unique base patient ID (P_xxxxx) over the 4 CSVs")
check("Test share of cases (abnormalities)", 0.20, round(float((ABN.split == "test").mean()), 3), "TCIA ('about 20%')",
      "test abnormality rows / all rows; tolerance ±0.03", abs_tol=0.03)
check("Pathology classes", PATHOLOGY_CLASSES, set(ABN.pathology), "CBIS-DDSM thesis", "unique pathology values")
check("Metadata CSV files present", set(CSV_FILES.values()), {p.name for p in CSV_PATHS.values()}, "TCIA",
      "downloaded file names")

if IDC is not None:
    check("DICOM 'subjects' (PatientID values)", 6671, int(IDC.PatientID.nunique()), "TCIA", "IDC index, whole collection")
    check("Images (DICOM instances) in collection", 10239, int(IDC.instanceCount.sum()), "TCIA", "IDC index")
    check("Series in collection", 6775, int(len(IDC)), "TCIA", "IDC index")
    check("Collection size, GB", 163.6, round(float(IDC.series_size_MB.sum() / 1000), 1), "TCIA",
          "IDC series_size_MB / 1000; tolerance ±2%", rel_tol=0.02)
    check("Full-image DICOM series referenced by CSVs", 3103, int(len(FULL_SERIES)), "Petrini 2025 Tab. 1 (images)",
          "distinct SeriesInstanceUID in 'image file path'")
    check("Referenced full-image series with malformed IDC PatientID", 0, IDC_PATIENTID_ANOMALY["series"],
          "not in PRD (data-quality check)", "known IDC defect, see 4.3; matched by UID instead")
else:
    for f, e in [("DICOM 'subjects' (PatientID values)", 6671), ("Images (DICOM instances) in collection", 10239),
                 ("Series in collection", 6775)]:
        check(f, e, None, "TCIA", "IDC route not used")

_tr = IMAGES[IMAGES.split == "train"]
check("Train images matched to a downloaded DICOM (%)", 100.0,
      None if INVENTORY is None or "train" not in set(INVENTORY.split) else round(100.0 * float(_tr.dicom_path.notna().mean()), 2),
      "PRD §5.4", "train images in the DICOM inventory / train images (test: checked at final evaluation)")

COUNTS = pd.DataFrame(checks)

# Reconciliation computed in code: image counts per PRD §4 (one row per split/patient/side/view) vs distinct series.
if IDC is not None:
    print("Reconciliation of image counts (distinct DICOM series → de-duplicated images):")
    for sp in ["train", "test"]:
        n_ser = int((FULL_SERIES.split == sp).sum())
        n_dup = int(((IMAGES.split == sp) & (IMAGES.n_series > 1)).sum())
        print(f"  {sp:5s}: {n_ser:,} series − {n_dup} images listed in both mass and calc CSVs (one extra series each) "
              f"= {n_ser - n_dup:,}  | de-duplicated images = {n_img(sp):,}  "
              f"{'✅ reconciles' if n_ser - n_dup == n_img(sp) else '❌ does not reconcile'}")

COUNTS.to_csv(cfg.results_dir / "counts_check.csv", index=False)
with pd.option_context("display.max_colwidth", 60):
    print(COUNTS.drop(columns=["how computed"]).to_string(index=False))
MISMATCHES = COUNTS[COUNTS.match == "❌"]
print(f"\n{len(COUNTS)} checks: {int((COUNTS.match == '✅').sum())} match, {len(MISMATCHES)} mismatch, "
      f"{int((COUNTS.match == 'n/a').sum())} not run. Saved to {cfg.results_dir / 'counts_check.csv'}")

### Review gate 1 — counts

Compare the counts table with PRD §2. Every ❌ must be explained before training code is written. Approval is pinned to
the reviewed set of mismatches (`GATE1_REVIEWED_MISMATCHES`): if a later run produces a different set (e.g. a new IDC
data release), the gate closes again.

**Decision (approved 2026-11-10, IDC data v25, idc-index 0.13.4):**

| Mismatch | Explanation | Consequence |
|---|---|---|
| Full-mammogram images total / train / test lower than Petrini Table 1 | Petrini's counts equal the number of distinct full-image DICOM series (that row ✅). 42 images (38 train, 4 test) are listed in both the mass and calc CSVs, each copy as its own series. PRD §4 collapses them to one image per (split, patient, side, view); the reconciliation printed above shows series − duplicates = images exactly. | Unit of prediction is the de-duplicated image. Milestone 7 verifies the duplicate pairs are pixel-identical. |
| 246 full-image series with malformed IDC PatientID | Calc-Test series whose DICOM PatientID is `P_xxxxx_SIDE_VIEW.dcm` with an empty SeriesDescription (source metadata defect). | Matched by Series/Study UID; only that exact pattern is accepted; split and patient come from the CSVs. |

In [ ]:
_mismatch_facts = set(MISMATCHES.fact)
_reviewed = set(cfg.GATE1_REVIEWED_MISMATCHES)
_msg = (f"{len(MISMATCHES)} mismatch(es) vs PRD §2: " + "; ".join(
    f"{r.fact}: expected {r.expected}, observed {r.observed}" for r in MISMATCHES.itertuples())
        if len(MISMATCHES) else "All counts match PRD §2.")
if cfg.GATE1_COUNTS_APPROVED and _mismatch_facts != _reviewed:
    raise ReviewGate("⛔ Review gate 1 re-closed: the mismatches differ from the reviewed set.\n"
                     f"  new: {sorted(_mismatch_facts - _reviewed)}\n  gone: {sorted(_reviewed - _mismatch_facts)}\n{_msg}")
print(_msg)
review_gate("Review gate 1 (metadata counts)", cfg.GATE1_COUNTS_APPROVED, _msg)

## 6. EDA

Class balance per split, image-size histogram, and sample images per class and abnormality type.
**Train split only for anything that touches pixels or DICOM headers** — the test split appears here only as metadata
counts. Figures are saved small to `results/figures/`.

In [ ]:
# ---- 6.0 Imaging helpers (shared by EDA, preprocessing and, later, the final evaluation)
import cv2
import pydicom
import matplotlib
import matplotlib.pyplot as plt

cv2.setNumThreads(0)  # leave parallelism to the process pool


def read_dicom_uint16(path):
    '''Decode a CBIS-DDSM DICOM to a 2-D uint16 array where higher = denser (brighter).'''
    ds = pydicom.dcmread(path)
    arr = ds.pixel_array
    assert arr.ndim == 2, f"{path}: expected a 2-D image, got shape {arr.shape}"
    assert int(ds.get("PixelRepresentation", 0)) == 0, f"{path}: signed pixel data is not expected"
    bits = int(ds.get("BitsStored", 16))
    photometric = str(ds.get("PhotometricInterpretation", ""))
    assert photometric in ("MONOCHROME1", "MONOCHROME2"), f"{path}: unexpected PhotometricInterpretation {photometric}"
    arr = arr.astype(np.uint16, copy=False)
    if photometric == "MONOCHROME1":  # PRD §5.5: invert so that every image has the same polarity
        arr = ((1 << bits) - 1 - arr).astype(np.uint16)
    return arr, {"photometric": photometric, "bits_stored": bits, "raw_min": int(arr.min()), "raw_max": int(arr.max())}


def to_uint8_preview(arr, long_side=256):
    h, w = arr.shape
    s = long_side / max(h, w)
    small = cv2.resize(arr, (max(1, round(w * s)), max(1, round(h * s))), interpolation=cv2.INTER_AREA)
    lo, hi = np.percentile(small, [0.5, 99.5])
    return np.clip((small.astype(np.float32) - lo) / max(hi - lo, 1) * 255, 0, 255).astype(np.uint8)


def save_fig(fig, name):
    p = cfg.figures_dir / name
    fig.savefig(p, dpi=90, bbox_inches="tight")
    print("figure →", p)
    return p


TRAIN_IMAGES = IMAGES[IMAGES.split == "train"].reset_index(drop=True)

In [ ]:
# ---- 6.1 Class balance per split and abnormality type (metadata only)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, sp in zip(axes, ["train", "test"]):
    t = pd.crosstab(IMAGES.loc[IMAGES.split == sp, "abn_type"], IMAGES.loc[IMAGES.split == sp, "label"])
    t = t.reindex(["mass", "calc", "both"]).fillna(0)
    t.columns = ["benign", "malignant"]
    t.plot.bar(ax=ax, rot=0, color=["#7aa6c2", "#c2577a"])
    ax.set_title(f"official {sp}: {int(t.values.sum())} images, {IMAGES.loc[IMAGES.split == sp, 'label'].mean():.1%} malignant")
    ax.set_xlabel("")
fig.tight_layout()
save_fig(fig, "eda_class_balance.png")
plt.show()
plt.close(fig)

In [ ]:
# ---- 6.2 Image sizes from train DICOM headers (cached on Drive so later sessions need no DICOM)
HDR_PATH = cfg.meta_dir / "train_dicom_headers.csv"
if HDR_PATH.exists():
    HDR = pd.read_csv(HDR_PATH)
    print(f"Loaded cached headers for {len(HDR):,} train images from {HDR_PATH}")
elif TRAIN_IMAGES.dicom_path.notna().all() and len(TRAIN_IMAGES):
    from concurrent.futures import ThreadPoolExecutor

    def _hdr(row):
        ds = pydicom.dcmread(row.dicom_path, stop_before_pixels=True)
        return {"image_key": row.image_key, "rows": int(ds.Rows), "cols": int(ds.Columns),
                "bits_stored": int(ds.BitsStored), "photometric": str(ds.PhotometricInterpretation),
                "transfer_syntax": str(ds.file_meta.TransferSyntaxUID)}

    with ThreadPoolExecutor(16) as ex:
        HDR = pd.DataFrame(list(ex.map(_hdr, TRAIN_IMAGES.itertuples())))
    HDR.to_csv(HDR_PATH, index=False)
    print(f"Read {len(HDR):,} train headers → {HDR_PATH}")
else:
    HDR = None
    print("ℹ️  No train DICOMs on disk and no cached headers — size histogram skipped.")

if HDR is not None:
    print(HDR[["photometric", "bits_stored", "transfer_syntax"]].value_counts().to_string())
    print(HDR[["rows", "cols"]].describe().round(0).to_string())
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    axes[0].hist(HDR.rows, bins=40); axes[0].set_title("rows (height, px)")
    axes[1].hist(HDR.cols, bins=40); axes[1].set_title("columns (width, px)")
    axes[2].hist(HDR.rows / HDR.cols, bins=40); axes[2].set_title("aspect (rows / cols)")
    fig.suptitle(f"Train full-mammogram sizes (n={len(HDR):,})")
    fig.tight_layout()
    save_fig(fig, "eda_image_sizes.png")
    plt.show()
    plt.close(fig)

In [ ]:
# ---- 6.3 Sample train images per class × abnormality type (small previews only)
_have_px = TRAIN_IMAGES.dicom_path.notna().all() and all(Path(p).exists() for p in TRAIN_IMAGES.dicom_path.head(5))
if _have_px:
    groups = [(lab, at) for lab in (0, 1) for at in ("mass", "calc", "both")]
    per = 3
    fig, axes = plt.subplots(len(groups), per, figsize=(per * 2.2, len(groups) * 2.9))
    rng = np.random.default_rng(cfg.seed)
    for gi, (lab, at) in enumerate(groups):
        pool = TRAIN_IMAGES[(TRAIN_IMAGES.label == lab) & (TRAIN_IMAGES.abn_type == at)]
        pick = pool.iloc[rng.choice(len(pool), size=min(per, len(pool)), replace=False)] if len(pool) else pool
        for j in range(per):
            ax = axes[gi, j]
            ax.axis("off")
            if j < len(pick):
                r = pick.iloc[j]
                arr, _ = read_dicom_uint16(r.dicom_path)
                ax.imshow(to_uint8_preview(arr), cmap="gray")
                ax.set_title(f"{'malig' if lab else 'benign'} · {at}\n{r.base_pid} {r.side[0]}-{r['view']}", fontsize=7)
    fig.tight_layout()
    save_fig(fig, "eda_samples_train.png")
    plt.show()
    plt.close(fig)
else:
    print("ℹ️  Train DICOMs not on this runtime — sample grid skipped (it is saved as "
          "results/figures/eda_samples_train.png by the session that downloads them).")

## 7. Preprocessing and caching (PRD §5.5–5.8)

Per train image: decode (honour `PhotometricInterpretation`, keep 16-bit) → segment the breast with Otsu on a 512 px
copy, keep the largest connected component → crop to its bounding box + margin → flip so the breast always sits on the
**left** (chest wall at the left edge) → resize to a 2048 px long side (aspect preserved, never upscaled) → 16-bit PNG.

* A manifest CSV records key, label, groups, path, original size, crop box, flip and raw min/max per image.
* The PNG directory and manifest are tarred to Drive with a completion marker; later sessions untar from Drive instead.
* QA: suspicious crops are listed, a contact sheet of crops is saved, and the 38 train mass/calc duplicate pairs are
  checked for pixel identity (the de-duplication in milestone 5 assumes they are the same image).
* **Test images are not processed here.** The same function runs on test in milestone 14, only when `FINAL_EVAL`.

Per-image z-score normalisation over breast pixels happens in the dataset (milestone 9), so the cache holds no
dataset-level statistics.

In [ ]:
# ---- 7.1 Preprocessing function (pure; runs in worker processes)


def segment_breast(img16, long_side, border_trim_frac, open_kernel_frac):
    '''Otsu + largest connected component on a downsampled copy. Returns (mask_small, scale).'''
    h, w = img16.shape
    s = long_side / max(h, w)
    small = cv2.resize(img16, (max(1, round(w * s)), max(1, round(h * s))), interpolation=cv2.INTER_AREA)
    lo, hi = np.percentile(small, [0.5, 99.5])
    u8 = np.clip((small.astype(np.float32) - lo) / max(hi - lo, 1) * 255, 0, 255).astype(np.uint8)
    _, th = cv2.threshold(u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    bh, bw = max(1, round(th.shape[0] * border_trim_frac)), max(1, round(th.shape[1] * border_trim_frac))
    th[:bh], th[-bh:], th[:, :bw], th[:, -bw:] = 0, 0, 0, 0  # scanner edges / film borders
    k = max(3, int(round(long_side * open_kernel_frac)) | 1)  # odd kernel, ≈15 px at 512
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((k, k), np.uint8))
    n, lab, stats, _ = cv2.connectedComponentsWithStats(th, connectivity=8)
    if n <= 1:
        return None, s
    k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    return (lab == k), s


def resize_long_side(img, long_side):
    h, w = img.shape
    if max(h, w) <= long_side:
        return img
    s = long_side / max(h, w)
    return cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))), interpolation=cv2.INTER_AREA)


def preprocess_one(job):
    '''job: dict(image_key, dicom_path, out_path, out_uncropped or None, params). Returns a manifest record.'''
    rec = {"image_key": job["image_key"]}
    try:
        p = job["params"]
        img, meta = read_dicom_uint16(job["dicom_path"])
        rec.update(meta)
        H, W = img.shape
        rec.update(orig_h=H, orig_w=W)
        mask, s = segment_breast(img, p["otsu_long_side"], p["border_trim_frac"], p["open_kernel_frac"])
        if mask is None:
            raise RuntimeError("no foreground component found")
        ys, xs = np.nonzero(mask)
        y0, y1, x0, x1 = ys.min() / s, (ys.max() + 1) / s, xs.min() / s, (xs.max() + 1) / s
        my, mx = p["crop_margin_frac"] * H, p["crop_margin_frac"] * W
        y0, y1 = int(max(0, y0 - my)), int(min(H, y1 + my))
        x0, x1 = int(max(0, x0 - mx)), int(min(W, x1 + mx))
        cols = mask.sum(axis=0)
        flip = bool(cols[len(cols) // 2:].sum() > cols[:len(cols) // 2].sum())  # breast mass on the right → flip
        rec.update(crop_y0=y0, crop_y1=y1, crop_x0=x0, crop_x1=x1, flipped=flip,
                   breast_frac_lowres=float(mask.mean()), crop_area_frac=float((y1 - y0) * (x1 - x0) / (H * W)))

        crop = img[y0:y1, x0:x1]
        if flip:
            crop = crop[:, ::-1]
        out = resize_long_side(np.ascontiguousarray(crop), p["cache_long_side"])
        assert cv2.imwrite(job["out_path"], out), f"cv2.imwrite failed for {job['out_path']}"
        rec.update(out_h=out.shape[0], out_w=out.shape[1], png_path=job["out_rel"])
        if job.get("out_uncropped"):
            full = img[:, ::-1] if flip else img
            assert cv2.imwrite(job["out_uncropped"], resize_long_side(np.ascontiguousarray(full), p["cache_long_side"]))
            rec["png_uncropped_path"] = job["out_uncropped_rel"]
        rec["error"] = None
    except Exception as e:  # recorded, then asserted on after the pool finishes
        rec["error"] = f"{type(e).__name__}: {e}"
    return rec

In [ ]:
# ---- 7.2 Build the train cache (or restore it from Drive)
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp


def restore_cache(split: str) -> pd.DataFrame:
    '''Copy + untar the PNG cache from Drive to local disk if needed; return its manifest.'''
    marker = json.loads(cfg.cache_marker(split).read_text())
    man = pd.read_csv(cfg.manifest_path(split))
    assert hashlib.sha256(cfg.manifest_path(split).read_bytes()).hexdigest() == marker["manifest_sha256"], \
        "manifest on Drive does not match the cache marker"
    missing = [p for p in man.png_path if not (cfg.png_dir / p).exists()]
    if missing:
        t0 = time.time()
        local_tar = cfg.local / cfg.cache_tar(split).name
        print(f"Restoring {split} cache: copying {cfg.cache_tar(split).stat().st_size / 1e9:.1f} GB tar from Drive …", flush=True)
        shutil.copy(cfg.cache_tar(split), local_tar)
        sh(f"tar -xf '{local_tar}' -C '{cfg.png_dir}'", check=True)
        local_tar.unlink()
        print(f"  restored in {(time.time() - t0) / 60:.1f} min")
    missing = [p for p in man.png_path if not (cfg.png_dir / p).exists()]
    assert not missing, f"{len(missing)} PNGs still missing after restore, e.g. {missing[:5]}"
    return man


PARAMS = {k: getattr(cfg, k) for k in ["otsu_long_side", "border_trim_frac", "open_kernel_frac", "crop_margin_frac",
                                       "cache_long_side"]}

if cache_complete("train"):
    MANIFEST = restore_cache("train")
    print(f"✅ Train cache {cfg.cache_version} ready: {len(MANIFEST):,} PNGs in {cfg.png_dir / 'train'}")
elif not cfg.RUN_PREPROCESS:
    raise RuntimeError("RUN_PREPROCESS is False but no complete train cache exists on Drive.")
else:
    assert TRAIN_IMAGES.dicom_path.notna().all(), "train DICOMs are required to build the cache (milestone 4)"
    out_dir = cfg.png_dir / "train"
    out_dir.mkdir(parents=True, exist_ok=True)
    if cfg.cache_uncropped:
        (cfg.png_dir / "train_uncropped").mkdir(parents=True, exist_ok=True)
    log_path = cfg.local / f"preprocess_train_{cfg.cache_version}.jsonl"  # resume within a session
    done = {}
    if log_path.exists():
        for line in log_path.read_text().splitlines():
            r = json.loads(line)
            if r.get("error") is None and (cfg.png_dir / r["png_path"]).exists():
                done[r["image_key"]] = r
    jobs = []
    for r in TRAIN_IMAGES.itertuples():
        if r.image_key in done:
            continue
        jobs.append({"image_key": r.image_key, "dicom_path": r.dicom_path, "params": PARAMS,
                     "out_path": str(out_dir / f"{r.image_key}.png"), "out_rel": f"train/{r.image_key}.png",
                     "out_uncropped": str(cfg.png_dir / "train_uncropped" / f"{r.image_key}.png") if cfg.cache_uncropped else None,
                     "out_uncropped_rel": f"train_uncropped/{r.image_key}.png"})
    print(f"Preprocessing {len(jobs):,} train images ({len(done):,} already done) with {cfg.n_workers} workers …", flush=True)
    t0 = time.time()
    ctx = mp.get_context("fork") if "fork" in mp.get_all_start_methods() else None
    with ProcessPoolExecutor(max_workers=cfg.n_workers, mp_context=ctx) as ex, open(log_path, "a") as log:
        futs = [ex.submit(preprocess_one, j) for j in jobs]
        for i, f in enumerate(as_completed(futs), 1):
            rec = f.result()
            log.write(json.dumps(rec) + "\n")
            done[rec["image_key"]] = rec
            if i % 100 == 0 or i == len(futs):
                el = time.time() - t0
                print(f"  {i:5d}/{len(futs)}  {el / 60:5.1f} min  ETA {el / i * (len(futs) - i) / 60:5.1f} min", flush=True)
    recs = pd.DataFrame(list(done.values()))
    errs = recs[recs.error.notna()]
    assert errs.empty, f"{len(errs)} images failed preprocessing:\n{errs[['image_key', 'error']].head(30).to_string()}"
    keep = ["image_key", "split", "base_pid", "side", "view", "label", "abn_type", "n_abnormalities", "pathologies",
            "breast_density", "assessment_max", "subtlety_max", "series_uid"]
    MANIFEST = TRAIN_IMAGES[keep].merge(recs.drop(columns=["error"]), on="image_key", how="left", validate="1:1")
    assert MANIFEST.png_path.notna().all() and len(MANIFEST) == len(TRAIN_IMAGES)
    print(f"Done: {len(MANIFEST):,} PNGs in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# ---- 7.3 QA: value ranges, suspicious crops, read-back, duplicate-pair identity, contact sheet
print("Raw value range (uint16):")
print(MANIFEST[["raw_min", "raw_max"]].describe().round(0).to_string())
print("Photometric:", MANIFEST.photometric.value_counts().to_dict(), "| flipped:", int(MANIFEST.flipped.sum()),
      f"of {len(MANIFEST)}")
print("Side vs flipped. DDSM films are not consistently oriented, so flips do NOT follow the side label exactly;\n"
      "the flip is decided from the breast mask (check the contact sheet: chest wall always on the left).")
print(pd.crosstab(MANIFEST.side, MANIFEST.flipped).to_string())

SUSPICIOUS = MANIFEST[(MANIFEST.crop_area_frac < 0.15) | (MANIFEST.crop_area_frac > 0.97) | (MANIFEST.breast_frac_lowres < 0.05)]
print(f"\nSuspicious crops (area <15% or >97% of the film, or tiny breast mask): {len(SUSPICIOUS)}")
if len(SUSPICIOUS):
    print(SUSPICIOUS[["image_key", "crop_area_frac", "breast_frac_lowres", "orig_h", "orig_w"]].head(20).to_string(index=False))

# Read-back: PNGs are 16-bit and the expected size
for r in MANIFEST.sample(min(5, len(MANIFEST)), random_state=cfg.seed).itertuples():
    a = cv2.imread(str(cfg.png_dir / r.png_path), cv2.IMREAD_UNCHANGED)
    assert a is not None and a.dtype == np.uint16 and a.shape == (r.out_h, r.out_w), f"bad PNG {r.png_path}"
print("✅ PNG read-back: uint16, sizes match the manifest")

# Mass/calc duplicate pairs (same split/patient/side/view, two series): must be the same pixels
_dups = TRAIN_IMAGES[TRAIN_IMAGES.n_series > 1]
if len(_dups) and INVENTORY is not None and all(Path(p).exists() for p in INVENTORY.dicom_path.head(3)):
    _inv = INVENTORY.set_index("series_uid").dicom_path
    same, diff = 0, []
    for r in _dups.itertuples():
        a_uid, b_uid = r.all_series_uids.split("|")[:2]
        a, _ = read_dicom_uint16(_inv[a_uid])
        b, _ = read_dicom_uint16(_inv[b_uid])
        if a.shape == b.shape and np.array_equal(a, b):
            same += 1
        else:
            diff.append((r.image_key, a.shape, b.shape,
                         None if a.shape != b.shape else int(np.abs(a.astype(np.int32) - b).max())))
    print(f"Mass/calc duplicate pairs (train): {same} pixel-identical, {len(diff)} different")
    assert not diff, ("De-duplication assumption broken — these duplicate pairs are not the same image:\n"
                      + "\n".join(map(str, diff[:20])))
else:
    print("ℹ️  Duplicate-pair pixel check skipped (DICOMs not on this runtime; it ran when the cache was built).")

# Contact sheet of random crops, with the crop box drawn on a low-res view of the original film
_sheet = MANIFEST.sample(min(12, len(MANIFEST)), random_state=cfg.seed + 1)
fig, axes = plt.subplots(2, len(_sheet), figsize=(1.6 * len(_sheet), 6.2), squeeze=False)
_inv_paths = INVENTORY.set_index("series_uid").dicom_path if INVENTORY is not None else {}
for j, r in enumerate(_sheet.itertuples()):
    png = cv2.imread(str(cfg.png_dir / r.png_path), cv2.IMREAD_UNCHANGED)
    axes[1, j].imshow(to_uint8_preview(png), cmap="gray")
    axes[1, j].set_title("cached", fontsize=7)
    dp = _inv_paths.get(r.series_uid) if len(_inv_paths) else None
    if dp is not None and Path(dp).exists():
        arr, _ = read_dicom_uint16(dp)
        prev = to_uint8_preview(arr)
        s = prev.shape[0] / r.orig_h
        axes[0, j].imshow(prev, cmap="gray")
        axes[0, j].add_patch(plt.Rectangle((r.crop_x0 * s, r.crop_y0 * s), (r.crop_x1 - r.crop_x0) * s,
                                           (r.crop_y1 - r.crop_y0) * s, fill=False, color="yellow", lw=1))
    axes[0, j].set_title(f"{r.base_pid} {r.side[0]}-{r.view}", fontsize=7)  # r is an itertuples namedtuple here
    for ax in axes[:, j]:
        ax.axis("off")
fig.tight_layout()
save_fig(fig, f"preprocess_contact_sheet_{cfg.cache_version}.png")
plt.show()
plt.close(fig)

In [ ]:
# ---- 7.4 Write manifest + tar to Drive with a completion marker (only when the cache was built this session)
if not cache_complete("train"):
    mpath = cfg.manifest_path("train")
    MANIFEST.to_csv(mpath, index=False)
    t0 = time.time()
    tar_tmp = cfg.cache_tar("train").with_suffix(".tar.part")
    print(f"Writing {cfg.cache_tar('train').name} to Drive …", flush=True)
    dirs = "train" + (" train_uncropped" if cfg.cache_uncropped else "")
    sh(f"tar -cf '{tar_tmp}' -C '{cfg.png_dir}' {dirs}", check=True)
    tar_tmp.rename(cfg.cache_tar("train"))
    marker = {"split": "train", "cache_version": cfg.cache_version, "n_images": int(len(MANIFEST)),
              "manifest_sha256": hashlib.sha256(mpath.read_bytes()).hexdigest(),
              "tar_bytes": cfg.cache_tar("train").stat().st_size, "params": PARAMS,
              "cache_uncropped": cfg.cache_uncropped, "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
              "data_route": DATA_ROUTE}
    cfg.cache_marker("train").write_text(json.dumps(marker, indent=2, default=str))
    print(f"✅ Cache on Drive: {marker['tar_bytes'] / 1e9:.1f} GB tar, manifest sha256 {marker['manifest_sha256'][:12]} "
          f"({(time.time() - t0) / 60:.1f} min)")
    if cfg.delete_dicom_after_cache:
        n_del = 0
        for u in INVENTORY.loc[INVENTORY.split == "train", "series_uid"]:
            shutil.rmtree(cfg.dicom_dir / u, ignore_errors=True)
            n_del += 1
        print(f"Deleted {n_del:,} train DICOM series from local disk (delete_dicom_after_cache=True).")
else:
    print("Cache already on Drive — nothing to write.")
MANIFEST_SHA256 = json.loads(cfg.cache_marker("train").read_text())["manifest_sha256"]

## 8. Splits and leakage asserts (PRD §6)

* Patients that appear in **both** official splits are dropped from train/validation (never from test); the count is
  reported.
* `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)` over the remaining official-train images, grouped by
  base patient ID, stratified by label. **Fold 0** is the development hold-out for ablations.
* Asserts: no patient in more than one fold, no train/val patient in test, both views of a breast in the same fold,
  every dev image has a cached PNG. Prevalence per fold is printed.
* Folds are computed from metadata only, so they do not change if preprocessing changes.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

TEST_PIDS = set(IMAGES.loc[IMAGES.split == "test", "base_pid"])
_train_all = IMAGES[IMAGES.split == "train"]
_dropped = _train_all[_train_all.base_pid.isin(CROSS_SPLIT_PIDS)]
DEV = _train_all[~_train_all.base_pid.isin(CROSS_SPLIT_PIDS)].sort_values("image_key").reset_index(drop=True)
print(f"Cross-split patients: {len(CROSS_SPLIT_PIDS)} → dropped {len(_dropped)} train images "
      f"({_dropped.label.sum()} malignant). Dev set: {len(DEV):,} images, {DEV.base_pid.nunique():,} patients.")

sgkf = StratifiedGroupKFold(n_splits=cfg.n_folds, shuffle=True, random_state=cfg.cv_seed)
DEV["fold"] = -1
for f, (_, va_idx) in enumerate(sgkf.split(DEV, DEV.label, groups=DEV.base_pid)):
    DEV.loc[va_idx, "fold"] = f

# ---- Leakage asserts
assert (DEV.fold >= 0).all(), "some images were not assigned a fold"
assert (DEV.groupby("base_pid").fold.nunique() == 1).all(), "a patient appears in more than one fold"
assert (DEV.groupby(["base_pid", "side"]).fold.nunique() == 1).all(), "the two views of a breast are in different folds"
assert not (set(DEV.base_pid) & TEST_PIDS), "a train/val patient also appears in the official test split"
assert DEV.image_key.is_unique
_missing_png = set(DEV.image_key) - set(MANIFEST.image_key)
assert not _missing_png, f"{len(_missing_png)} dev images have no cached PNG, e.g. {sorted(_missing_png)[:5]}"
print("✅ Leakage asserts passed: patients disjoint across folds and from test; breasts never split across folds; "
      "every dev image has a PNG.")

FOLD_TABLE = DEV.groupby("fold").agg(images=("image_key", "size"), patients=("base_pid", "nunique"),
                                     malignant=("label", "sum"), prevalence=("label", "mean"),
                                     mass=("abn_type", lambda s: (s == "mass").sum()),
                                     calc=("abn_type", lambda s: (s == "calc").sum()),
                                     both=("abn_type", lambda s: (s == "both").sum()))
FOLD_TABLE["prevalence"] = FOLD_TABLE.prevalence.round(3)
print(FOLD_TABLE.to_string())

FOLDS_PATH = cfg.meta_dir / f"folds_{cfg.cache_version}.csv"
DEV[["image_key", "base_pid", "side", "view", "label", "abn_type", "fold"]].to_csv(FOLDS_PATH, index=False)
FOLDS_SHA256 = hashlib.sha256(FOLDS_PATH.read_bytes()).hexdigest()
SPLIT_SUMMARY = {"cross_split_patients": len(CROSS_SPLIT_PIDS), "dropped_train_images": int(len(_dropped)),
                 "dev_images": int(len(DEV)), "dev_patients": int(DEV.base_pid.nunique()),
                 "test_images": int((IMAGES.split == "test").sum()), "n_folds": cfg.n_folds, "cv_seed": cfg.cv_seed,
                 "folds_sha256": FOLDS_SHA256, "manifest_sha256": MANIFEST_SHA256,
                 "per_fold": FOLD_TABLE.reset_index().to_dict(orient="records")}
(cfg.results_dir / "splits.json").write_text(json.dumps(SPLIT_SUMMARY, indent=2, default=int))
print(f"Folds → {FOLDS_PATH} (sha256 {FOLDS_SHA256[:12]}); summary → {cfg.results_dir / 'splits.json'}")

## 9. Datasets, dataloaders and augmentation

* Images come from the local 16-bit PNG cache (milestone 7). Each is resized to the experiment's `(H, W)` with aspect
  preserved and **letterboxed**: chest wall on the left edge, padding on the right and top/bottom.
* **Per-image normalisation** (PRD §5): z-score over breast pixels (Otsu mask of that image), so no statistics are shared
  between images, folds or splits.
* Train-only augmentation (PRD §8, Shen et al. ranges): horizontal + vertical flips, rotation ±25°, scale 0.8–1.2,
  brightness/contrast/gamma jitter. No elastic warps.
* Input is one channel; the model's stem sums the pretrained RGB filters (`timm` `in_chans=1`).

In [ ]:
for _mod, _pip in {"timm": "timm", "albumentations": "albumentations", "scipy": "scipy"}.items():
    if importlib.util.find_spec(_mod) is None:
        print("Installing", _pip)
        print(sh(f"{sys.executable} -m pip install -q {_pip}", check=True)[-300:])

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm
import albumentations as A

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16  # T4 has no bf16 → fp16 + scaler
else:
    AMP_DTYPE = None  # CPU: full precision (smoke tests only)
print("device:", DEVICE, "| AMP dtype:", AMP_DTYPE, "| timm", timm.__version__, "| albumentations", A.__version__)


def letterbox(img, H, W):
    '''Resize preserving aspect ratio to fit (H, W); chest wall stays on the left; returns (canvas, scale, (top, left)).'''
    h, w = img.shape
    s = min(H / h, W / w)
    nh, nw = max(1, round(h * s)), max(1, round(w * s))
    r = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)
    canvas = np.zeros((H, W), dtype=img.dtype)
    top = (H - nh) // 2
    canvas[top:top + nh, :nw] = r
    return canvas, s, (top, 0)


def breast_zscore(x):
    '''x: float32 [0,1]. z-score using mean/std of the breast pixels (Otsu) of this image only.'''
    u8 = (np.clip(x, 0, 1) * 255).astype(np.uint8)
    t, _ = cv2.threshold(u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = u8 > t
    if m.mean() < 0.01:
        m = np.ones_like(m)
    mu, sd = float(x[m].mean()), float(x[m].std())
    return (x - mu) / max(sd, 1e-6)


def make_train_aug():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Affine(rotate=(-25, 25), scale=(0.8, 1.2), p=0.8),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
        A.RandomGamma(gamma_limit=(80, 120), p=0.5),
    ])


class MammoDataset(Dataset):
    def __init__(self, df, image_size, train: bool, path_col="png_path", aug=None):
        self.df = df.reset_index(drop=True)
        self.H, self.W = image_size
        self.train = train
        self.path_col = path_col
        self.aug = aug

    def __len__(self):
        return len(self.df)

    def load(self, i):
        r = self.df.iloc[i]
        img = cv2.imread(str(cfg.png_dir / r[self.path_col]), cv2.IMREAD_UNCHANGED)
        assert img is not None and img.dtype == np.uint16, f"bad PNG {r[self.path_col]}"
        img, _, _ = letterbox(img, self.H, self.W)
        return img.astype(np.float32) / 65535.0

    def __getitem__(self, i):
        x = self.load(i)
        if self.train and self.aug is not None:
            x = self.aug(image=x)["image"]
        x = breast_zscore(x).astype(np.float32)
        r = self.df.iloc[i]
        return torch.from_numpy(x)[None], torch.tensor(float(r.label)), i


def make_loader(df, image_size, train, batch_size, path_col="png_path", sampler="none", aug=True):
    ds = MammoDataset(df, image_size, train, path_col, make_train_aug() if (train and aug) else None)
    smp = None
    if train and sampler == "balanced":
        y = df.label.values
        w = np.where(y == 1, 0.5 / max(y.sum(), 1), 0.5 / max((1 - y).sum(), 1))
        smp = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), num_samples=len(df), replacement=True)
    nw = cfg.n_workers if DEVICE.type == "cuda" else 0
    return DataLoader(ds, batch_size=batch_size, shuffle=(train and smp is None), sampler=smp, drop_last=train,
                      num_workers=nw, pin_memory=DEVICE.type == "cuda", persistent_workers=nw > 0,
                      prefetch_factor=4 if nw > 0 else None)


DEV_M = DEV.merge(MANIFEST.drop(columns=[c for c in DEV.columns if c in MANIFEST.columns and c != "image_key"]),
                  on="image_key", how="left", validate="1:1")
assert DEV_M.png_path.notna().all()
print(f"Dev images with PNGs: {len(DEV_M):,}; uncropped variant available: {'png_uncropped_path' in DEV_M.columns}")

In [ ]:
# ---- 9.1 Augmentation preview grid (one train image, original + 7 augmented views, normalised)
_ds = MammoDataset(DEV_M[DEV_M.fold != cfg.fold], (1152, 896), train=True, aug=make_train_aug())
_i = int(np.random.default_rng(cfg.seed).integers(len(_ds)))
_base = breast_zscore(_ds.load(_i))
fig, axes = plt.subplots(1, 8, figsize=(14, 2.9))
axes[0].imshow(_base, cmap="gray", vmin=-2.5, vmax=2.5); axes[0].set_title("original", fontsize=8)
for j in range(1, 8):
    x, _, _ = _ds[_i]
    axes[j].imshow(x[0].numpy(), cmap="gray", vmin=-2.5, vmax=2.5); axes[j].set_title(f"aug {j}", fontsize=8)
for ax in axes:
    ax.axis("off")
fig.tight_layout()
save_fig(fig, "augmentation_preview.png")
plt.show()
plt.close(fig)
print(f"normalised tensor: shape {tuple(x.shape)}, mean {x.mean():.2f}, std {x.std():.2f}")

## 10. Models

* `MammoNet`: `timm` backbone (ImageNet weights, 1-channel stem) → feature map → pooling (`avg`, `gem`, or gated
  attention-based **MIL** pooling over feature-map positions) → dropout → one logit.
* `PatchNet`: the same backbone with a 5-class head (background, benign/malignant mass, benign/malignant calcification)
  for the Shen-style patch pre-training ablation (Stage 2). Its backbone weights load straight into `MammoNet`.

In [ ]:
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        x = x.float()  # pow/root are unstable in bf16
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p), x.shape[-2:]).pow(1.0 / self.p).flatten(1)


class AttnMILPool(nn.Module):
    '''Gated attention pooling (Ilse et al. 2018) over the h×w feature positions, treated as instances.'''
    def __init__(self, c, hidden=256):
        super().__init__()
        self.v = nn.Linear(c, hidden)
        self.u = nn.Linear(c, hidden)
        self.w = nn.Linear(hidden, 1)
        self.last_attention = None

    def forward(self, x):
        b, c, h, w = x.shape
        inst = x.flatten(2).transpose(1, 2)                      # (B, hw, C)
        a = self.w(torch.tanh(self.v(inst)) * torch.sigmoid(self.u(inst)))  # (B, hw, 1)
        a = torch.softmax(a.float(), dim=1).to(inst.dtype)
        self.last_attention = a.detach().view(b, h, w)
        return (a * inst).sum(1)


def make_pool(kind, c):
    if kind == "avg":
        return nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(1))
    if kind == "gem":
        return GeM()
    if kind == "attn":
        return AttnMILPool(c)
    raise ValueError(kind)


class MammoNet(nn.Module):
    def __init__(self, backbone, pool="avg", drop_rate=0.3, pretrained=True, n_out=1):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained, in_chans=1, num_classes=0, global_pool="")
        c = self.backbone.num_features
        self.pool = make_pool(pool, c)
        self.drop = nn.Dropout(drop_rate)
        self.head = nn.Linear(c, n_out)

    def forward(self, x):
        z = self.head(self.drop(self.pool(self.backbone.forward_features(x))))
        return z.squeeze(-1) if z.shape[-1] == 1 else z

    def param_groups(self, lr_backbone, lr_head, weight_decay):
        bb = [p for p in self.backbone.parameters() if p.requires_grad]
        hd = [p for n, p in self.named_parameters() if not n.startswith("backbone.") and p.requires_grad]
        return [{"params": bb, "lr": lr_backbone, "weight_decay": weight_decay},
                {"params": hd, "lr": lr_head, "weight_decay": weight_decay}]


def PatchNet(backbone, pretrained=True):
    return MammoNet(backbone, pool="avg", drop_rate=0.2, pretrained=pretrained, n_out=5)


def build_model(exp, pretrained=True):
    m = MammoNet(exp.backbone, exp.pool, exp.drop_rate, pretrained=pretrained)
    return m.to(DEVICE).to(memory_format=torch.channels_last)


_m = MammoNet("resnet18", "attn", pretrained=False)
_y = _m(torch.randn(2, 1, 256, 192))
assert _y.shape == (2,), _y.shape
assert PatchNet("resnet18", pretrained=False)(torch.randn(2, 1, 224, 224)).shape == (2, 5)
del _m, _y
print("✅ model factory shape checks passed")

## 11. Training utilities

* **Experiment config** (`Exp`): every knob an ablation can turn. `train_hash` covers only what affects training, so an
  eval-only change (hflip TTA) re-uses the parent's trained checkpoint.
* bf16 autocast (fp16 + GradScaler on GPUs without bf16), `channels_last`, TF32, optional `torch.compile`.
* **Batch size from free GPU memory** (probe, cached per GPU/backbone/size), gradient accumulation to the effective batch.
* AdamW, backbone LR < head LR, 1-epoch linear warm-up then cosine decay; early stopping on validation AUC.
* Optional EMA of weights (evaluated with the EMA weights).
* **Checkpoint every epoch to Drive** (`last.pt`, `best.pt`: model, optimiser, scheduler, scaler, EMA, epoch, history,
  RNG state) with automatic resume; a config-hash mismatch refuses to resume.
* Metrics: ROC AUC with **2,000 patient-level bootstrap** resamples (95% CI), PR-AUC, sensitivity at 80/90% specificity,
  specificity at 90% sensitivity. Every run appends a row to `experiments.csv`; completed runs are skipped on re-run.

In [ ]:
import dataclasses
from dataclasses import replace
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve


@dataclass(frozen=True)
class Exp:
    name: str
    stage: str
    hypothesis: str
    parent: str = None
    backbone: str = "efficientnet_b3"
    image_size: tuple = (1152, 896)       # (H, W)
    input: str = "cropped"                # cropped | uncropped
    init: str = "imagenet"                # imagenet | patch (Stage 2)
    pool: str = "avg"                     # avg | gem | attn
    drop_rate: float = 0.3
    loss: str = "bce"                     # bce | focal
    pos_weight: bool = False              # BCE pos_weight = n_neg / n_pos of the training fold
    focal_gamma: float = 2.0
    sampler: str = "none"                 # none | balanced
    aug: bool = True
    epochs: int = 30
    patience: int = 8
    lr_backbone: float = 1e-4
    lr_head: float = 1e-3
    weight_decay: float = 1e-3
    warmup_epochs: float = 1.0
    effective_batch: int = 16
    max_batch: int = 64
    ema: bool = False
    ema_decay: float = 0.999
    compile: bool = False
    tta_hflip: bool = False               # evaluation only
    seed: int = 42

    EVAL_ONLY = ("tta_hflip",)
    META = ("name", "stage", "hypothesis", "parent")

    def _hash(self, exclude):
        d = {k: v for k, v in dataclasses.asdict(self).items() if k not in exclude}
        return hashlib.sha256(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:12]

    @property
    def train_hash(self): return self._hash(self.META + self.EVAL_ONLY)
    @property
    def exp_hash(self): return self._hash(self.META)


def exp_diff(a: Exp, b: Exp) -> dict:
    '''Fields that differ between parent a and child b (ignoring metadata).'''
    da, db = dataclasses.asdict(a), dataclasses.asdict(b)
    return {k: (da[k], db[k]) for k in da if k not in Exp.META and da[k] != db[k]}


def size_for_long_side(L, aspect=896 / 1152, mult=32):
    return (int(L), int(round(L * aspect / mult) * mult))


# ---------------- metrics
def auc_rank(y, s):
    y = np.asarray(y); s = np.asarray(s)
    n1 = y.sum(); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(s)
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def bootstrap_auc(y, s, groups, n_boot=2000, seed=0):
    '''Patient-level bootstrap: resample patients with replacement, take all their images.'''
    y, s, groups = np.asarray(y), np.asarray(s), np.asarray(groups)
    uniq, inv = np.unique(groups, return_inverse=True)
    idx_by_g = [np.flatnonzero(inv == g) for g in range(len(uniq))]
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(n_boot):
        pick = rng.integers(0, len(uniq), len(uniq))
        ii = np.concatenate([idx_by_g[g] for g in pick])
        a = auc_rank(y[ii], s[ii])
        if not np.isnan(a):
            out.append(a)
    out = np.array(out)
    return float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5)), out


def sens_at_spec(y, s, spec):
    fpr, tpr, _ = roc_curve(y, s)
    ok = fpr <= 1 - spec
    return float(tpr[ok].max()) if ok.any() else 0.0


def spec_at_sens(y, s, sens):
    fpr, tpr, _ = roc_curve(y, s)
    ok = tpr >= sens
    return float(1 - fpr[ok].min()) if ok.any() else 0.0


def binary_metrics(y, s, groups, n_boot=2000, seed=0):
    lo, hi, _ = bootstrap_auc(y, s, groups, n_boot, seed)
    return {"auc": float(roc_auc_score(y, s)), "auc_ci95": [lo, hi], "pr_auc": float(average_precision_score(y, s)),
            "sens_at_spec80": sens_at_spec(y, s, 0.80), "sens_at_spec90": sens_at_spec(y, s, 0.90),
            "spec_at_sens90": spec_at_sens(y, s, 0.90), "n": int(len(y)), "n_pos": int(np.sum(y)),
            "n_patients": int(len(np.unique(groups)))}


_yy = np.r_[np.zeros(50), np.ones(50)]; _ss = _yy + np.random.default_rng(0).normal(0, 1, 100)
assert abs(auc_rank(_yy, _ss) - roc_auc_score(_yy, _ss)) < 1e-9
print("✅ rank-AUC matches sklearn")

In [ ]:
# ---------------- losses, EMA, schedule, checkpoints
class BinaryFocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, y):
        bce = F.binary_cross_entropy_with_logits(logits, y, reduction="none")
        p_t = torch.exp(-bce)
        return ((1 - p_t) ** self.gamma * bce).mean()


class EMA:
    def __init__(self, model, decay):
        import copy
        self.decay = decay
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(self.decay).add_(msd[k].detach(), alpha=1 - self.decay)
            else:
                v.copy_(msd[k])


def warmup_cosine(total_steps, warmup_steps):
    def f(step):
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))
    return f


def rng_state():
    return {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def set_rng_state(st):
    random.setstate(st["python"]); np.random.set_state(st["numpy"]); torch.set_rng_state(st["torch"])
    if st.get("cuda") is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(st["cuda"])


def atomic_save(obj, path):
    tmp = Path(str(path) + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def file_sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def autocast():
    if AMP_DTYPE is None:
        return torch.autocast("cpu", enabled=False)
    return torch.autocast("cuda", dtype=AMP_DTYPE)


# ---------------- batch size from free GPU memory
BATCH_PROBE_PATH = cfg.results_dir / "batch_probe.json"


def require_training_gpu():
    '''Stop before any training on a CUDA GPU that is not an A100 (CPU smoke tests are unaffected).'''
    name = GPU_INFO.get("name") or ""
    if DEVICE.type == "cuda" and "A100" not in name and not cfg.allow_non_a100_training:
        raise RuntimeError(
            f"⛔ GPU is {name}, not an A100 — stopping before training. Data stages (download, cache, folds) are done "
            "and cached on Drive. Re-run on an A100 runtime, or set cfg.allow_non_a100_training = True.")


def probe_batch_size(exp):
    require_training_gpu()
    if DEVICE.type != "cuda":
        return min(exp.max_batch, 4)
    key = f"{GPU_INFO.get('name')}|{exp.backbone}|{exp.image_size}|{exp.pool}|{exp.compile}"
    cache = json.loads(BATCH_PROBE_PATH.read_text()) if BATCH_PROBE_PATH.exists() else {}
    if key in cache:
        return cache[key]
    H, W = exp.image_size
    found = 1
    for bs in [b for b in (64, 48, 32, 24, 16, 12, 8, 6, 4, 2, 1) if b <= exp.max_batch]:
        model = opt = x = loss = None
        try:
            torch.cuda.empty_cache()
            model = build_model(exp, pretrained=False).train()
            opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
            x = torch.randn(bs, 1, H, W, device=DEVICE).to(memory_format=torch.channels_last)
            for _ in range(2):
                with autocast():
                    loss = model(x).float().mean()
                loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
            torch.cuda.synchronize()
            found = bs
            break
        except torch.cuda.OutOfMemoryError:
            pass
        finally:
            del model, opt, x, loss
            torch.cuda.empty_cache()
    cache[key] = found
    BATCH_PROBE_PATH.write_text(json.dumps(cache, indent=2))
    print(f"  batch probe {key}: {found}")
    return found

In [ ]:
# ---------------- one fold of one experiment
EXPERIMENTS_CSV = cfg.results_dir / "experiments.csv"
EXP_REGISTRY = {}  # name → Exp, filled as experiments are defined


def run_dir(exp, fold):
    d = cfg.ckpt_dir / f"{exp.backbone}_{exp.train_hash}" / f"fold{fold}"
    d.mkdir(parents=True, exist_ok=True)
    return d


def git_commit():
    out = sh("git rev-parse --short HEAD")
    return out if re.fullmatch(r"[0-9a-f]{7,40}", out or "") else "unknown"


@torch.no_grad()
def predict(model, loader, tta_hflip=False):
    model.eval()
    probs = np.zeros(len(loader.dataset), dtype=np.float64)
    for x, _, idx in loader:
        x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        with autocast():
            p = torch.sigmoid(model(x).float())
            if tta_hflip:
                p = 0.5 * (p + torch.sigmoid(model(torch.flip(x, dims=[3])).float()))
        probs[idx.numpy()] = p.cpu().numpy()
    return probs


def train_fold(exp: Exp, fold: int, dev=None, verbose=True):
    '''Train (or resume, or reuse) exp on dev[fold != f], validate on dev[fold == f]. Returns (metrics, val_df).'''
    require_training_gpu()
    dev = DEV_M if dev is None else dev
    path_col = "png_path" if exp.input == "cropped" else "png_uncropped_path"
    if path_col not in dev.columns:
        raise RuntimeError(f"{exp.name}: input '{exp.input}' needs cfg.cache_uncropped=True and a rebuilt cache")
    tr, va = dev[dev.fold != fold].reset_index(drop=True), dev[dev.fold == fold].reset_index(drop=True)
    assert not (set(tr.base_pid) & set(va.base_pid)), "patient leakage between train and validation"
    rd = run_dir(exp, fold)
    done_path, last_path, best_path = rd / "done.json", rd / "last.pt", rd / "best.pt"
    seed_everything(exp.seed + fold)

    def batch_plan():
        # Never exceed the configured effective batch: the probe gives the largest batch that *fits*; the batch used is
        # min(fit, effective_batch), and gradient accumulation makes up the rest (PRD §8: effective batch 16–32).
        b = min(probe_batch_size(exp), exp.effective_batch)
        return b, max(1, math.ceil(exp.effective_batch / b))

    if last_path.exists() and not done_path.exists():
        _ck = torch.load(last_path, map_location="cpu", weights_only=False)
        _plan = batch_plan()
        if (_ck.get("batch"), _ck.get("accum")) != _plan:
            print(f"  ⚠️  {rd.parent.name}/{rd.name}: checkpoint was trained with batch {_ck.get('batch')} × accum "
                  f"{_ck.get('accum')}, not {_plan[0]} × {_plan[1]} — set aside as *.stale; training restarts.")
            for _p in (last_path, best_path):
                if _p.exists():
                    _p.rename(_p.with_name(_p.name + ".stale"))
        del _ck

    model = build_model(exp, pretrained=not last_path.exists())
    if exp.init == "patch" and not last_path.exists():
        sd = torch.load(patch_backbone_path(exp, fold), map_location="cpu")
        model.backbone.load_state_dict(sd, strict=True)
        print(f"  initialised backbone from patch model ({patch_backbone_path(exp, fold).parent.name})")

    if not done_path.exists():
        bs, accum = batch_plan()
        train_loader = make_loader(tr, exp.image_size, True, bs, path_col, exp.sampler, exp.aug)
        val_loader = make_loader(va, exp.image_size, False, bs * 2, path_col)
        steps_per_epoch = math.ceil(len(train_loader) / accum)
        opt = torch.optim.AdamW(model.param_groups(exp.lr_backbone, exp.lr_head, exp.weight_decay))
        sched = torch.optim.lr_scheduler.LambdaLR(
            opt, warmup_cosine(steps_per_epoch * exp.epochs, int(steps_per_epoch * exp.warmup_epochs)))
        scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))
        ema = EMA(model, exp.ema_decay) if exp.ema else None
        if exp.loss == "focal":
            crit = BinaryFocalLoss(exp.focal_gamma)
        else:
            pw = torch.tensor([(1 - tr.label.mean()) / tr.label.mean()], device=DEVICE) if exp.pos_weight else None
            crit = nn.BCEWithLogitsLoss(pos_weight=pw)
        fwd = torch.compile(model) if exp.compile else model

        start_epoch, best_auc, best_epoch, history, gpu_seconds = 0, -1.0, -1, [], 0.0
        if last_path.exists():
            ck = torch.load(last_path, map_location="cpu", weights_only=False)
            assert ck["train_hash"] == exp.train_hash, "checkpoint belongs to a different configuration"
            model.load_state_dict(ck["model"]); opt.load_state_dict(ck["optimizer"]); sched.load_state_dict(ck["scheduler"])
            scaler.load_state_dict(ck["scaler"])
            if ema is not None and ck.get("ema") is not None:
                ema.module.load_state_dict(ck["ema"])
            start_epoch, best_auc, best_epoch = ck["epoch"] + 1, ck["best_auc"], ck["best_epoch"]
            history, gpu_seconds = ck["history"], ck["gpu_seconds"]
            set_rng_state(ck["rng"])
            print(f"  resumed {rd.parent.name}/{rd.name} at epoch {start_epoch} (best AUC {best_auc:.4f} @ {best_epoch})")
        if verbose:
            print(f"  {exp.name} fold {fold}: train {len(tr)} / val {len(va)} | batch {bs} × accum {accum} | "
                  f"{steps_per_epoch} steps/epoch | {rd}")

        for epoch in range(start_epoch, exp.epochs):
            if epoch - best_epoch > exp.patience and best_epoch >= 0:
                print(f"  early stop at epoch {epoch} (no improvement for {exp.patience} epochs)")
                break
            t0 = time.time()
            model.train()
            run_loss, n_seen = 0.0, 0
            opt.zero_grad(set_to_none=True)
            for it, (x, y, _) in enumerate(train_loader):
                x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
                y = y.to(DEVICE, non_blocking=True)
                with autocast():
                    logits = fwd(x).float()
                loss = crit(logits, y) / accum
                scaler.scale(loss).backward()
                if (it + 1) % accum == 0 or it + 1 == len(train_loader):
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
                    if ema is not None:
                        ema.update(model)
                run_loss += loss.item() * accum * len(y); n_seen += len(y)
            eval_model = ema.module if ema is not None else model
            p = predict(eval_model, val_loader)
            auc = float(roc_auc_score(va.label, p))
            val_loss = float(F.binary_cross_entropy(torch.tensor(p).clamp(1e-6, 1 - 1e-6), torch.tensor(va.label.values, dtype=torch.float64)))
            dt = time.time() - t0
            gpu_seconds += dt
            history.append({"epoch": epoch, "train_loss": run_loss / max(n_seen, 1), "val_loss": val_loss, "val_auc": auc,
                            "lr_backbone": opt.param_groups[0]["lr"], "seconds": dt})
            improved = auc > best_auc
            if improved:
                best_auc, best_epoch = auc, epoch
                atomic_save({"model": eval_model.state_dict(), "exp": dataclasses.asdict(exp), "train_hash": exp.train_hash,
                             "epoch": epoch, "val_auc": auc}, best_path)
            atomic_save({"model": model.state_dict(), "optimizer": opt.state_dict(), "scheduler": sched.state_dict(),
                         "scaler": scaler.state_dict(), "ema": ema.module.state_dict() if ema is not None else None,
                         "epoch": epoch, "best_auc": best_auc, "best_epoch": best_epoch, "history": history,
                         "gpu_seconds": gpu_seconds, "rng": rng_state(), "train_hash": exp.train_hash,
                         "batch": bs, "accum": accum, "exp": dataclasses.asdict(exp)}, last_path)
            print(f"  ep {epoch:2d}  loss {history[-1]['train_loss']:.4f}  val_loss {val_loss:.4f}  val_AUC {auc:.4f}"
                  f"{' *' if improved else ''}  {dt / 60:.1f} min", flush=True)
        done = {"best_auc_raw": best_auc, "best_epoch": best_epoch, "epochs_run": len(history), "history": history,
                "gpu_minutes": gpu_seconds / 60, "batch": bs, "accum": accum, "gpu": GPU_INFO.get("name"),
                "best_sha256": file_sha256(best_path)}
        done_path.write_text(json.dumps(done, indent=2))
        del opt, sched, train_loader
    done = json.loads(done_path.read_text())

    # Final evaluation of best.pt on the validation fold (with TTA if this experiment asks for it)
    model.load_state_dict(torch.load(best_path, map_location="cpu", weights_only=False)["model"])
    bs = probe_batch_size(exp)
    p = predict(model, make_loader(va, exp.image_size, False, bs * 2, path_col), tta_hflip=exp.tta_hflip)
    va = va.assign(prob=p, exp=exp.name, fold=fold)
    met = binary_metrics(va.label.values, p, va.base_pid.values, seed=exp.seed + fold)
    met.update({k: done[k] for k in ["best_epoch", "epochs_run", "gpu_minutes", "batch", "accum", "gpu", "best_sha256"]})
    met["history"] = done["history"]
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return met, va

In [ ]:
# ---------------- experiment bookkeeping (experiments.csv, curves, summary.json)
def load_experiments():
    return pd.read_csv(EXPERIMENTS_CSV) if EXPERIMENTS_CSV.exists() else pd.DataFrame()


def plot_curves(exp, fold, history):
    h = pd.DataFrame(history)
    if h.empty:
        return
    fig, ax = plt.subplots(1, 2, figsize=(8, 2.6))
    ax[0].plot(h.epoch, h.train_loss, label="train"); ax[0].plot(h.epoch, h.val_loss, label="val"); ax[0].legend()
    ax[0].set_title("loss"); ax[1].plot(h.epoch, h.val_auc); ax[1].set_title("val AUC")
    fig.suptitle(f"{exp.name} · fold {fold}", fontsize=9); fig.tight_layout()
    save_fig(fig, f"curves_{exp.name}_fold{fold}.png"); plt.show(); plt.close(fig)


def run_experiment(exp: Exp, fold: int = 0):
    '''Idempotent: a finished (exp_hash, fold) row in experiments.csv is returned without re-running.'''
    EXP_REGISTRY[exp.name] = exp
    df = load_experiments()
    if len(df):
        hit = df[(df.exp_hash == exp.exp_hash) & (df.fold == fold) & (df.status == "done")]
        if len(hit):
            r = hit.iloc[-1]
            same = "" if r.exp == exp.name else f" (identical configuration already run as {r.exp})"
            print(f"↺ {exp.name} fold {fold}: cached{same} — val AUC {r.val_auc:.4f} [{r.auc_ci_lo:.4f}, {r.auc_ci_hi:.4f}]")
            return r.to_dict()
    parent = EXP_REGISTRY.get(exp.parent) if exp.parent else None
    changed = exp_diff(parent, exp) if parent else {}
    print(f"▶ {exp.name} (stage {exp.stage}, fold {fold}) — {exp.hypothesis}\n  change vs {exp.parent}: {changed or '—'}")
    met, va = train_fold(exp, fold)
    pred_dir = cfg.results_dir / "preds"
    pred_dir.mkdir(exist_ok=True)
    va[["image_key", "base_pid", "label", "abn_type", "view", "fold", "exp", "prob"]].to_csv(
        pred_dir / f"val_{exp.name}_fold{fold}.csv", index=False)
    plot_curves(exp, fold, met.pop("history"))
    row = {"timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "exp": exp.name, "stage": exp.stage,
           "parent": exp.parent, "changed": json.dumps(changed, default=str), "hypothesis": exp.hypothesis,
           "fold": fold, "split": f"5-fold patient-grouped CV in official train, fold {fold} (validation)",
           "val_auc": met["auc"], "auc_ci_lo": met["auc_ci95"][0], "auc_ci_hi": met["auc_ci95"][1],
           "pr_auc": met["pr_auc"], "sens_at_spec90": met["sens_at_spec90"], "n_val": met["n"], "n_val_pos": met["n_pos"],
           "best_epoch": met["best_epoch"], "epochs_run": met["epochs_run"], "gpu": met["gpu"],
           "gpu_minutes": round(met["gpu_minutes"], 2), "batch": met["batch"], "accum": met["accum"],
           "exp_hash": exp.exp_hash, "train_hash": exp.train_hash, "best_sha256": met["best_sha256"],
           "manifest_sha256": MANIFEST_SHA256, "folds_sha256": FOLDS_SHA256, "git_commit": git_commit(),
           "config": json.dumps(dataclasses.asdict(exp), default=str), "status": "done"}
    df = pd.concat([load_experiments(), pd.DataFrame([row])], ignore_index=True)
    df.to_csv(EXPERIMENTS_CSV, index=False)
    print(f"✔ {exp.name} fold {fold}: val AUC {row['val_auc']:.4f} [{row['auc_ci_lo']:.4f}, {row['auc_ci_hi']:.4f}] "
          f"({row['gpu_minutes']:.0f} GPU-min) → {EXPERIMENTS_CSV.name}")
    write_summary()
    return row


def write_summary(extra=None):
    df = load_experiments()
    s = {"updated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "n_configurations_evaluated": int(df.exp_hash.nunique()) if len(df) else 0,
         "total_gpu_minutes": float(df.gpu_minutes.sum()) if len(df) else 0.0,
         "runs": df[["exp", "stage", "fold", "val_auc", "auc_ci_lo", "auc_ci_hi", "gpu_minutes"]].to_dict(orient="records") if len(df) else []}
    if extra:
        s.update(extra)
    (cfg.results_dir / "summary.json").write_text(json.dumps(s, indent=2, default=str))

## 12. Experiments (fold 0)

Each stage is a controlled experiment on **fold 0**, changing one thing at a time; a change is kept only if it raises
fold-0 validation AUC. Every configuration evaluated is a row in `experiments.csv` (the README reports how many).
Fold-0 AUCs are selection numbers on ~480 images and are optimistic after many comparisons — the headline is the
5-fold result in milestone 13.

### Stage 0 — pipeline baseline
ResNet-50, short side 512 px, a few epochs. Goal: prove data, labels and metrics work end to end. Expect a weak score;
a suspiciously high one points to leakage.

In [ ]:
S0 = Exp(name="s0_resnet50_512", stage="0", hypothesis="Pipeline works end to end; expect a weak but non-random AUC.",
         backbone="resnet50", image_size=(672, 512), epochs=6, patience=6, effective_batch=16)
r0 = run_experiment(S0, fold=cfg.fold)
if r0["val_auc"] > 0.85:
    print("⚠️  Stage 0 AUC is suspiciously high for a 6-epoch 512 px baseline — check for leakage before going on.")
elif r0["val_auc"] < 0.55:
    print("⚠️  Stage 0 AUC is near chance — check labels, normalisation and the data pipeline.")

### Stage 0 diagnostic (pipeline sanity checks)

Run because the first Stage 0 baseline was near chance (2026-11-11, L4: fold-0 val AUC 0.5433 [0.4774, 0.6070]).
**Development data only** (official train split). This is not an experiment: nothing is written to `experiments.csv`,
and its numbers are diagnostics, not results. Output → `results/diagnostics_s0.json`; the cell skips when it exists.

* **D1 labels:** labels returned by the dataset/loader equal `images.csv` for the same `image_key`; prevalence per fold.
* **D2 inputs:** per-image statistics of normalised tensors (with and without augmentation), orientation (breast on
  the left), and a saved grid of loader inputs with their labels.
* **D3 overfit:** ResNet-50 at 672×512 must memorise 64 training images (32/32, no augmentation). If train loss does
  not approach 0, the training code is broken.
* **D4 BatchNorm:** AUC of the D3 model on those 64 images in `eval()` vs `train()` mode. If `eval()` is much lower,
  the evaluation path, not the data, loses the signal.
* **D5 linear probe:** frozen ImageNet ResNet-50 features + logistic regression, trained on folds ≠ 0 and scored on
  fold 0. Tells whether 512 px inputs carry usable signal before any fine-tuning.

In [ ]:
DIAG_PATH = cfg.results_dir / "diagnostics_s0.json"
if cfg.RUN_S0_DIAGNOSTIC and not DIAG_PATH.exists():
    require_training_gpu()
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    DIAG = {"exp": S0.name, "image_size": list(S0.image_size), "fold": cfg.fold, "gpu": GPU_INFO.get("name")}
    _tr = DEV_M[DEV_M.fold != cfg.fold].reset_index(drop=True)
    _va = DEV_M[DEV_M.fold == cfg.fold].reset_index(drop=True)
    _t0 = time.time()

    # ---- D1 labels: loader labels vs images.csv, by image_key
    _img = pd.read_csv(cfg.meta_dir / "images.csv").set_index("image_key")
    _ds = MammoDataset(_tr, S0.image_size, train=False)
    _idx = np.random.default_rng(0).choice(len(_ds), size=min(200, len(_ds)), replace=False)
    _bad = []
    for i in _idx:
        _, y, j = _ds[int(i)]
        k = _tr.image_key[int(j)]
        if int(j) != int(i) or int(y) != int(_img.at[k, "label"]):
            _bad.append(k)
    assert not _bad, f"D1: loader label/index disagrees with images.csv for {len(_bad)} images: {_bad[:10]}"
    assert (DEV_M.set_index("image_key").label == _img.label.reindex(DEV_M.image_key).values).all(), \
        "D1: DEV_M labels disagree with images.csv"
    DIAG["d1_prevalence_by_fold"] = DEV_M.groupby("fold").label.mean().round(3).to_dict()
    print(f"✅ D1 labels: {len(_idx)} sampled loader items match images.csv; malignant share by fold "
          f"{DIAG['d1_prevalence_by_fold']}")

    # ---- D2 inputs: normalised tensor statistics, orientation, saved grid
    def _stats(ds, n=32):
        rows = []
        for i in range(n):
            x = ds[i][0][0].numpy()
            h, w = x.shape
            rows.append({"mean": x.mean(), "std": x.std(), "min": x.min(), "max": x.max(),
                         "bg_frac": float((x < x.min() + 1e-3).mean()),
                         "left_minus_right": float(x[:, : w // 2].mean() - x[:, w // 2:].mean())})
        return pd.DataFrame(rows)
    _s_plain = _stats(MammoDataset(_tr, S0.image_size, train=False))
    _s_aug = _stats(MammoDataset(_tr, S0.image_size, train=True, aug=make_train_aug()))
    for nm, s in [("no aug", _s_plain), ("aug", _s_aug)]:
        print(f"D2 inputs ({nm}, 32 images): " + ", ".join(f"{c} {s[c].mean():.2f}±{s[c].std():.2f}" for c in s.columns))
    DIAG["d2_plain"] = _s_plain.mean().round(3).to_dict()
    DIAG["d2_aug"] = _s_aug.mean().round(3).to_dict()
    DIAG["d2_breast_left_share"] = float((_s_plain.left_minus_right > 0).mean())
    _dl = make_loader(_tr, S0.image_size, True, 8, sampler="none", aug=True)
    xb, yb, ib = next(iter(_dl))
    fig, axes = plt.subplots(1, 8, figsize=(14, 2.9))
    for a, x, y, i in zip(axes, xb, yb, ib):
        a.imshow(x[0].numpy(), cmap="gray", vmin=-2.5, vmax=2.5); a.axis("off")
        a.set_title(f"y={int(y)} {_tr.abn_type[int(i)]}\n{_tr.image_key[int(i)][-14:]}", fontsize=7)
    fig.tight_layout(); save_fig(fig, "diag_s0_loader_batch.png"); plt.show(); plt.close(fig)
    del _dl

    # ---- D3 overfit 64 training images (no augmentation) + D4 BatchNorm train/eval gap
    seed_everything(0)
    _pos = _tr[_tr.label == 1].sample(32, random_state=0)
    _neg = _tr[_tr.label == 0].sample(32, random_state=0)
    _small = pd.concat([_pos, _neg]).reset_index(drop=True)
    _sl = DataLoader(MammoDataset(_small, S0.image_size, train=False), batch_size=16, shuffle=True, drop_last=False,
                     num_workers=cfg.n_workers if DEVICE.type == "cuda" else 0)
    _el = DataLoader(MammoDataset(_small, S0.image_size, train=False), batch_size=16, shuffle=False,
                     num_workers=cfg.n_workers if DEVICE.type == "cuda" else 0)
    _m = build_model(S0, pretrained=True)
    _opt = torch.optim.AdamW(_m.param_groups(S0.lr_backbone, S0.lr_head, S0.weight_decay))
    _crit = nn.BCEWithLogitsLoss()
    _curve = []
    for ep in range(40):
        _m.train(); tot, nseen = 0.0, 0
        for x, y, _ in _sl:
            x = x.to(DEVICE).to(memory_format=torch.channels_last); y = y.to(DEVICE)
            with autocast():
                lo = _crit(_m(x).float(), y)
            _opt.zero_grad(set_to_none=True); lo.backward(); _opt.step()
            tot += lo.item() * len(y); nseen += len(y)
        _curve.append(round(tot / max(nseen, 1), 4))
        if ep % 5 == 4:
            print(f"  D3 overfit epoch {ep + 1:2d}: train loss {_curve[-1]:.4f}", flush=True)

    @torch.no_grad()
    def _probs(model, loader, train_mode):
        model.train(train_mode)
        out = []
        for x, _, _ in loader:
            with autocast():
                out.append(torch.sigmoid(model(x.to(DEVICE).to(memory_format=torch.channels_last)).float()).cpu())
        return torch.cat(out).numpy()
    _ys = _small.label.values
    _auc_eval = float(roc_auc_score(_ys, _probs(_m, _el, False)))
    _auc_train = float(roc_auc_score(_ys, _probs(_m, _el, True)))
    DIAG.update({"d3_overfit_loss_curve": _curve, "d3_final_loss": _curve[-1],
                 "d4_auc_eval_mode": round(_auc_eval, 4), "d4_auc_train_mode": round(_auc_train, 4)})
    print(f"{'✅' if _curve[-1] < 0.1 else '❌'} D3 overfit: train loss {_curve[0]:.4f} → {_curve[-1]:.4f} "
          f"(a working pipeline goes below 0.1)")
    # Only eval() falling well below train() is a problem (evaluation loses what training learned); the reverse is
    # normal, because train() normalises with the statistics of each small batch.
    print(f"{'✅' if _auc_eval >= _auc_train - 0.1 else '❌'} D4 BatchNorm: AUC on the 64 memorised images "
          f"eval() {_auc_eval:.4f} vs train() {_auc_train:.4f}")
    del _m, _opt, _sl, _el
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    # ---- D5 linear probe on frozen ImageNet features (no fine-tuning, no augmentation)
    _fm = timm.create_model(S0.backbone, pretrained=True, in_chans=1, num_classes=0).to(DEVICE).eval()

    @torch.no_grad()
    def _feats(df):
        dl = make_loader(df, S0.image_size, False, 32)
        F_ = np.zeros((len(df), _fm.num_features), dtype=np.float32)
        for x, _, idx in dl:
            with autocast():
                F_[idx.numpy()] = _fm(x.to(DEVICE)).float().cpu().numpy()
        return F_
    _Ftr, _Fva = _feats(_tr), _feats(_va)
    _best = None
    for C in (0.001, 0.01, 0.1, 1.0):
        # C is picked by AUC on fold 0 itself, so this probe AUC is optimistic — a diagnostic, not a result.
        clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000))
        clf.fit(_Ftr, _tr.label.values)
        a = float(roc_auc_score(_va.label.values, clf.predict_proba(_Fva)[:, 1]))
        print(f"  D5 linear probe C={C:<6}: fold-{cfg.fold} AUC {a:.4f}")
        if _best is None or a > _best[1]:
            _best = (C, a)
    DIAG.update({"d5_probe_best_C": _best[0], "d5_probe_auc_optimistic": round(_best[1], 4)})
    del _fm, _Ftr, _Fva
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    DIAG["minutes"] = round((time.time() - _t0) / 60, 1)
    DIAG_PATH.write_text(json.dumps(DIAG, indent=2, default=float))
    print(f"Diagnostics → {DIAG_PATH} ({DIAG['minutes']} min)")
elif DIAG_PATH.exists():
    print(f"↺ Stage 0 diagnostics already run: {DIAG_PATH}")
    print(DIAG_PATH.read_text())
else:
    print("ℹ️  Stage 0 diagnostic skipped (cfg.RUN_S0_DIAGNOSTIC is False).")

### Stage 0b — longer baseline (one change: 6 → 20 epochs)
The diagnostic found no pipeline bug, but the 6-epoch fine-tune underfit (train loss 0.69 → 0.67) and scored below a
frozen-feature linear probe. Same setup, 20 epochs; patience is unchanged.

In [ ]:
S0b = replace(S0, name="s0b_resnet50_512_e20", parent=S0.name, epochs=20,
              hypothesis="Stage 0 underfit in 6 epochs; 20 epochs should lift fold-0 AUC clearly above chance.")
r0b = run_experiment(S0b, fold=cfg.fold)

### Review gate 2 — baseline

Check the Stage 0 curves and AUC: losses fall, AUC is clearly above 0.5 and not suspiciously high, batch size and GPU
minutes look sane. Then set `GATE2_BASELINE_APPROVED = True`.

In [ ]:
review_gate("Review gate 2 (Stage 0 baseline)", cfg.GATE2_BASELINE_APPROVED,
            f"Stage 0 fold-{cfg.fold} validation AUC {r0['val_auc']:.4f} [{r0['auc_ci_lo']:.4f}, {r0['auc_ci_hi']:.4f}], "
            f"best epoch {r0['best_epoch']}, {r0['gpu_minutes']:.1f} GPU-min.\n"
            f"Stage 0b (20 epochs) validation AUC {r0b['val_auc']:.4f} [{r0b['auc_ci_lo']:.4f}, {r0b['auc_ci_hi']:.4f}], "
            f"best epoch {r0b['best_epoch']}, {r0b['gpu_minutes']:.1f} GPU-min.")

### torch.compile check
`torch.compile` is kept only if it measurably speeds up training steps (PRD §8). Timed once on the Stage 1 setup.

In [ ]:
COMPILE_PATH = cfg.results_dir / "compile_decision.json"


def time_steps(exp, n=20):
    bs = probe_batch_size(exp)
    model = build_model(exp, pretrained=False).train()
    fwd = torch.compile(model) if exp.compile else model
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    x = torch.randn(bs, 1, *exp.image_size, device=DEVICE).to(memory_format=torch.channels_last)
    for i in range(n + 5):  # 5 warm-up steps (compilation happens here)
        if i == 5:
            torch.cuda.synchronize(); t0 = time.time()
        with autocast():
            loss = fwd(x).float().mean()
        loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    del model, fwd, opt, x
    torch.cuda.empty_cache()
    return (time.time() - t0) / n / bs


_s1_probe = Exp(name="probe", stage="-", hypothesis="-")
if COMPILE_PATH.exists():
    USE_COMPILE = json.loads(COMPILE_PATH.read_text())["use_compile"]
elif DEVICE.type == "cuda":
    t_eager = time_steps(_s1_probe)
    try:
        t_comp = time_steps(replace(_s1_probe, compile=True))
    except Exception as e:
        print("torch.compile failed:", type(e).__name__, str(e)[:200]); t_comp = float("inf")
    USE_COMPILE = bool(t_comp < 0.9 * t_eager)  # keep only for a ≥10% speed-up
    COMPILE_PATH.write_text(json.dumps({"eager_s_per_img": t_eager, "compiled_s_per_img": t_comp,
                                        "use_compile": USE_COMPILE, "gpu": GPU_INFO.get("name")}, indent=2))
else:
    USE_COMPILE = False
print("torch.compile:", USE_COMPILE, json.loads(COMPILE_PATH.read_text()) if COMPILE_PATH.exists() else "")

### Stage 1 — direct whole-image classifier (DC)
EfficientNet-B3 (strongest single-view backbone in Petrini et al.) at 1152×896, then 1536×1184 if memory allows.

In [ ]:
SELECTION_LOG = []


def select(best_row, best_exp, cand_row, cand_exp, why):
    keep = cand_row["val_auc"] > best_row["val_auc"]
    SELECTION_LOG.append({"candidate": cand_exp.name, "vs": best_exp.name, "cand_auc": cand_row["val_auc"],
                          "best_auc": best_row["val_auc"], "kept": bool(keep), "reason": why})
    (cfg.results_dir / "selection_log.json").write_text(json.dumps(SELECTION_LOG, indent=2))
    print(f"  {'KEEP' if keep else 'drop'} {cand_exp.name}: {cand_row['val_auc']:.4f} vs {best_exp.name} {best_row['val_auc']:.4f}")
    return (cand_row, cand_exp) if keep else (best_row, best_exp)


S1 = Exp(name="s1_effb3_1152", stage="1", parent="s0_resnet50_512", compile=USE_COMPILE,
         hypothesis="EfficientNet-B3 at 1152×896 is a strong single-view DC baseline (Petrini 2025).")
EXP_REGISTRY[S0.name] = S0
r1 = run_experiment(S1, fold=cfg.fold)
BEST_ROW, BEST = r1, S1

S1b = replace(S1, name="s1_effb3_1536", parent=S1.name, image_size=size_for_long_side(1536),
              hypothesis="Higher resolution preserves calcification detail; Petrini saw AUC drop when halving 1152×896.")
if DEVICE.type == "cuda" and probe_batch_size(S1b) >= 2:
    BEST_ROW, BEST = select(BEST_ROW, BEST, run_experiment(S1b, fold=cfg.fold), S1b, "resolution 1536")
else:
    print("ℹ️  1536×1184 does not fit with batch ≥ 2 on this GPU — skipped.")

### Stage 2 — patch pre-training (Shen-style ablation), behind `RUN_PATCH_PRETRAIN`

224 px patches at the whole-image training scale, from **this fold's training images only**: patches around each ROI
(high overlap) labelled benign/malignant × mass/calcification, plus an equal number of background patches from breast
tissue outside all ROIs. The patch model's backbone then initialises the whole-image model, compared with Stage 1.

**Ablation row only (decided 2026-11-12).** Stage 2 was run after Stage 3 and the 5-fold CV, so its result is
logged and compared with its Stage 1 parent but does not replace the selected configuration.

ROI masks are downloaded here (train split only). CBIS-DDSM ROI series can hold both the mask and the cropped lesion,
so the mask is identified as the instance whose size equals the full image; size mismatches are counted and reported.

In [ ]:
PATCH_SIZE = 224


def patch_backbone_path(exp, fold):
    key = hashlib.sha256(f"{exp.backbone}|{exp.image_size}|{cfg.cache_version}|{FOLDS_SHA256}|v1".encode()).hexdigest()[:10]
    d = cfg.ckpt_dir / f"patch_{exp.backbone}_{key}" / f"fold{fold}"
    d.mkdir(parents=True, exist_ok=True)
    return d / "patch_backbone.pt"


def roi_mask_table():
    '''One row per TRAIN abnormality with its ROI series; downloads the series and identifies the mask instance.'''
    tr = ABN[(ABN.split == "train")].copy()
    tr["image_key"] = "train_" + tr.base_pid + "_" + tr.side + "_" + tr.view
    tr = tr[tr.image_key.isin(set(DEV_M.image_key))]
    tr["roi_series"] = tr.roi_mask_file_path.map(lambda p: parse_dicom_path(p)[2])
    tr["crop_series"] = tr.cropped_image_file_path.map(lambda p: parse_dicom_path(p)[2])
    assert tr.roi_series.notna().all(), "unparseable ROI mask paths"
    uids = sorted(set(tr.roi_series) | set(tr.crop_series))
    todo = [u for u in uids if not series_on_disk(u)]
    if todo:
        assert set(todo) <= set(IDC.SeriesInstanceUID), "ROI series missing from IDC"
        print(f"Downloading {len(todo):,} ROI/crop series …", flush=True)
        for i in range(0, len(todo), cfg.download_chunk_series):
            idc_client.download_from_selection(downloadDir=str(cfg.dicom_dir), seriesInstanceUID=todo[i:i + cfg.download_chunk_series],
                                               dirTemplate="%SeriesInstanceUID", use_s5cmd_sync=True,
                                               show_progress_bar=False, quiet=True)
    man = MANIFEST.set_index("image_key")
    paths, mism = [], 0
    for r in tr.itertuples():
        H, W = int(man.at[r.image_key, "orig_h"]), int(man.at[r.image_key, "orig_w"])
        cands = []
        for u in {r.roi_series, r.crop_series}:
            for f in series_on_disk(u):
                ds = pydicom.dcmread(f, stop_before_pixels=True)
                cands.append((int(ds.Rows), int(ds.Columns), str(f)))
        exact = [c for c in cands if (c[0], c[1]) == (H, W)]
        if len(exact) >= 1:
            paths.append(exact[0][2])
        else:
            mism += 1
            paths.append(max(cands, key=lambda c: c[0] * c[1])[2] if cands else None)
    tr["mask_path"] = paths
    print(f"ROI masks: {len(tr):,} abnormalities; {mism} without a same-size mask instance (largest instance used, resized)")
    assert tr.mask_path.notna().all()
    return tr


def mask_to_train_coords(mask_full, rec, image_size):
    '''Apply the cached crop/flip/resize, then the training letterbox, to a full-resolution mask.'''
    m = (mask_full > 0).astype(np.uint8)
    if m.shape != (rec.orig_h, rec.orig_w):
        m = cv2.resize(m, (int(rec.orig_w), int(rec.orig_h)), interpolation=cv2.INTER_NEAREST)
    m = m[int(rec.crop_y0):int(rec.crop_y1), int(rec.crop_x0):int(rec.crop_x1)]
    if rec.flipped:
        m = m[:, ::-1]
    m = cv2.resize(np.ascontiguousarray(m), (int(rec.out_w), int(rec.out_h)), interpolation=cv2.INTER_NEAREST)
    return letterbox(m, *image_size)[0]


def extract_patches(exp, fold, rois, n_per_roi=6, seed=0):
    '''Patches from dev images with fold != `fold` only. Returns float16 array (N,224,224), labels, patient ids.'''
    rng = np.random.default_rng(seed + fold)
    tr_keys = set(DEV_M.loc[DEV_M.fold != fold, "image_key"])
    rois = rois[rois.image_key.isin(tr_keys)]
    man = MANIFEST.set_index("image_key")
    X, Y, G = [], [], []
    half = PATCH_SIZE // 2
    cls = {("mass", "BENIGN"): 1, ("mass", "BENIGN_WITHOUT_CALLBACK"): 1, ("mass", "MALIGNANT"): 2,
           ("calc", "BENIGN"): 3, ("calc", "BENIGN_WITHOUT_CALLBACK"): 3, ("calc", "MALIGNANT"): 4}
    for n, (key, g) in enumerate(rois.groupby("image_key")):
        rec = man.loc[key]
        img = cv2.imread(str(cfg.png_dir / rec.png_path), cv2.IMREAD_UNCHANGED)
        img = breast_zscore(letterbox(img, *exp.image_size)[0].astype(np.float32) / 65535.0)
        Hh, Ww = img.shape
        union = np.zeros((Hh, Ww), np.uint8)
        for r in g.itertuples():
            m = mask_to_train_coords(pydicom.dcmread(r.mask_path).pixel_array, rec, exp.image_size)
            union |= m
            ys, xs = np.nonzero(m)
            if len(ys) == 0:
                continue
            for _ in range(n_per_roi):
                j = rng.integers(len(ys))
                cy = int(np.clip(ys[j] + rng.integers(-32, 33), half, Hh - half))
                cx = int(np.clip(xs[j] + rng.integers(-32, 33), half, Ww - half))
                X.append(img[cy - half:cy + half, cx - half:cx + half].astype(np.float16))
                Y.append(cls[(r.source, r.pathology)]); G.append(r.base_pid)
        # background: same number as ROI patches from this image, breast tissue, no ROI overlap
        need = n_per_roi * len(g)
        tissue = img > np.percentile(img, 40)
        for _ in range(need * 20):
            if need == 0:
                break
            cy, cx = int(rng.integers(half, Hh - half)), int(rng.integers(half, Ww - half))
            if tissue[cy, cx] and union[cy - half:cy + half, cx - half:cx + half].sum() == 0:
                X.append(img[cy - half:cy + half, cx - half:cx + half].astype(np.float16)); Y.append(0); G.append(g.base_pid.iloc[0])
                need -= 1
        if (n + 1) % 200 == 0:
            print(f"  patches from {n + 1} images: {len(X):,}", flush=True)
    return np.stack(X), np.array(Y), np.array(G)


def patch_pretrain(exp, fold, epochs=8, batch=128, lr=2e-4):
    out = patch_backbone_path(exp, fold)
    if not out.exists():
        require_training_gpu()
    if out.exists():
        print(f"↺ patch backbone cached: {out}")
        return out
    rois = roi_mask_table()
    X, Y, G = extract_patches(exp, fold, rois)
    print(f"Patches fold {fold}: {len(X):,}  class counts {np.bincount(Y, minlength=5).tolist()}")
    pids = np.unique(G); rng = np.random.default_rng(fold)
    hold = set(rng.choice(pids, max(1, len(pids) // 10), replace=False))  # patch-val = 10% of TRAINING patients
    is_va = np.array([g in hold for g in G])
    model = PatchNet(exp.backbone).to(DEVICE).to(memory_format=torch.channels_last)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()
    Xt, Yt = torch.from_numpy(X[~is_va]), torch.from_numpy(Y[~is_va])
    Xv, Yv = torch.from_numpy(X[is_va]), torch.from_numpy(Y[is_va])
    best = -1
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xt))
        for i in range(0, len(perm), batch):
            b = perm[i:i + batch]
            x = Xt[b].float()[:, None].to(DEVICE)
            if torch.rand(1) < 0.5: x = torch.flip(x, [3])
            if torch.rand(1) < 0.5: x = torch.flip(x, [2])
            x = torch.rot90(x, int(torch.randint(4, (1,))), [2, 3]).contiguous(memory_format=torch.channels_last)
            with autocast():
                loss = crit(model(x).float(), Yt[b].to(DEVICE))
            loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
        model.eval(); correct = 0
        with torch.no_grad():
            for i in range(0, len(Xv), batch * 2):
                x = Xv[i:i + batch * 2].float()[:, None].to(DEVICE).contiguous(memory_format=torch.channels_last)
                with autocast():
                    correct += (model(x).float().argmax(1).cpu() == Yv[i:i + batch * 2]).sum().item()
        acc = correct / max(1, len(Xv))
        print(f"  patch ep {ep}: patch-val accuracy {acc:.3f}", flush=True)
        if acc > best:
            best = acc
            atomic_save(model.backbone.state_dict(), out)
    (out.parent / "patch_metrics.json").write_text(json.dumps({"best_patch_val_acc": best, "n_patches": int(len(X)),
                                                               "class_counts": np.bincount(Y, minlength=5).tolist()}))
    return out


if cfg.RUN_PATCH_PRETRAIN:
    S2 = replace(BEST, name=f"s2_patchinit_{BEST.name}", stage="2", parent=BEST.name, init="patch",
                 hypothesis="Patch-level lesion pre-training (Shen 2019) gives the whole-image model better features; "
                            "Petrini found no significant gain (p=0.47).")
    patch_pretrain(S2, cfg.fold)
    # Ablation row only (decided in chat 2026-11-12): Stage 2 ran after Stage 3 and the 5-fold CV were already done on
    # the Stage 1 winner, so it is logged and compared with its parent but never replaces BEST. Feeding it into the
    # greedy selection would invalidate every later stage. This departs from the PRD's greedy rule and is reported.
    r2 = run_experiment(S2, fold=cfg.fold)
    _d2 = r2["val_auc"] - BEST_ROW["val_auc"]
    print(f"Stage 2 ablation (not used for selection): {S2.name} fold-{cfg.fold} AUC {r2['val_auc']:.4f} "
          f"[{r2['auc_ci_lo']:.4f}, {r2['auc_ci_hi']:.4f}] vs {BEST.name} {BEST_ROW['val_auc']:.4f} (Δ {_d2:+.4f})")
    STAGE2_ABLATION = {"exp": S2.name, "vs": BEST.name, "val_auc": r2["val_auc"], "ci95": [r2["auc_ci_lo"], r2["auc_ci_hi"]],
                       "parent_val_auc": BEST_ROW["val_auc"], "delta": _d2, "used_for_selection": False}
else:
    print("Stage 2 skipped (RUN_PATCH_PRETRAIN=False).")

### Stage 3 — one change at a time
Each ablation changes **one** field of the current best configuration; it replaces the best only if fold-0 AUC rises
(greedy, in the order of `STAGE3_RUN`). Completed runs are cached, so tags can be added to `STAGE3_RUN` later without
re-running anything.

**Prioritised subset (decided 2026-11-11):** the full list is about 18–25 GPU-hours at 1536 px on an L4, so the first
pass runs the changes aimed at what Stage 1 showed (overfitting: validation loss rises long before AUC peaks; small
lesions in a large image): EMA, stronger weight decay, GeM and attention pooling, then EfficientNet-B4 and
ConvNeXt-Tiny, and flip TTA last so it applies to the final configuration.

In [ ]:
STAGE3 = [
    ("res_896", dict(image_size=size_for_long_side(896)), "Lower resolution (896 long side) — checks the resolution trend."),
    ("bb_effb4", dict(backbone="efficientnet_b4"), "Larger EfficientNet capacity helps at this resolution."),
    ("bb_convnext_tiny", dict(backbone="convnext_tiny"), "A modern ConvNet backbone transfers better."),
    ("bb_convnext_small", dict(backbone="convnext_small"), "More ConvNeXt capacity helps."),
    ("bb_resnet50", dict(backbone="resnet50"), "Shen's backbone at full resolution."),
    ("uncropped", dict(input="uncropped"), "Breast cropping (vs Shen's uncropped input) matters."),
    ("pool_gem", dict(pool="gem"), "GeM pooling emphasises small salient regions (calcifications)."),
    ("pool_attn", dict(pool="attn"), "Attention-MIL pooling focuses on the lesion among mostly-normal tissue."),
    ("loss_focal", dict(loss="focal"), "Focal loss focuses training on hard examples."),
    ("bce_posweight", dict(pos_weight=True), "Re-weighting the minority class improves ranking."),
    ("sampler_balanced", dict(sampler="balanced"), "Balanced sampling instead of loss re-weighting."),
    ("wd_1e-4", dict(weight_decay=1e-4), "Weaker weight decay (search range 1e-4…1e-2)."),
    ("wd_1e-2", dict(weight_decay=1e-2), "Stronger weight decay regularises the small dataset."),
    ("ema", dict(ema=True), "EMA of weights smooths noisy small-batch updates."),
    ("tta_hflip", dict(tta_hflip=True), "Horizontal-flip TTA (evaluation only; reuses the trained checkpoint)."),
]

STAGE3_RUN = ("ema", "wd_1e-2", "pool_gem", "pool_attn", "bb_effb4", "bb_convnext_tiny", "tta_hflip")
_stage3_by_tag = {t: (t, c, h) for t, c, h in STAGE3}
assert set(STAGE3_RUN) <= set(_stage3_by_tag), f"unknown Stage 3 tags: {set(STAGE3_RUN) - set(_stage3_by_tag)}"
print("Stage 3 runs, in order:", ", ".join(STAGE3_RUN),
      "| not run this pass:", ", ".join(t for t in _stage3_by_tag if t not in STAGE3_RUN))

for tag, change, hyp in [_stage3_by_tag[t] for t in STAGE3_RUN]:
    cand = replace(BEST, name=f"s3_{tag}", stage="3", parent=BEST.name, hypothesis=hyp, **change)
    if cand.input == "uncropped" and "png_uncropped_path" not in DEV_M.columns:
        print(f"ℹ️  s3_{tag} skipped: set cache_uncropped=True and bump cache_version to build the uncropped cache.")
        continue
    if cand.exp_hash == BEST.exp_hash:
        print(f"ℹ️  s3_{tag} skipped: identical to the current best.")
        continue
    BEST_ROW, BEST = select(BEST_ROW, BEST, run_experiment(cand, fold=cfg.fold), cand, tag)

print(f"\nCurrent best on fold {cfg.fold}: {BEST.name}  val AUC {BEST_ROW['val_auc']:.4f} "
      f"[{BEST_ROW['auc_ci_lo']:.4f}, {BEST_ROW['auc_ci_hi']:.4f}]")
(cfg.results_dir / "best_config.json").write_text(json.dumps(dataclasses.asdict(BEST), indent=2, default=str))
write_summary({"best_fold0": {"exp": BEST.name, "val_auc": BEST_ROW["val_auc"],
                              "ci95": [BEST_ROW["auc_ci_lo"], BEST_ROW["auc_ci_hi"]]}})

In [ ]:
# ---- Ablation table (every configuration evaluated on validation)
_ex = load_experiments()
if len(_ex):
    _ex = _ex[_ex.fold == cfg.fold]
    _tab = _ex[["exp", "stage", "parent", "changed", "val_auc", "auc_ci_lo", "auc_ci_hi", "best_epoch", "gpu_minutes"]]
    print(f"{_ex.exp_hash.nunique()} configurations evaluated on fold {cfg.fold} "
          f"({_ex.gpu_minutes.sum() / 60:.1f} GPU-hours). Selection numbers — optimistic; see 5-fold CV for the headline.")
    print(_tab.round(4).to_string(index=False, max_colwidth=50))

## 13. 5-fold cross-validation of the chosen configuration (PRD §6, §7 Stage 4, §9)

The configuration selected on fold 0 (`BEST`) is trained on **all 5 folds** behind `RUN_CV`. Fold 0 is not re-trained:
an identical configuration hash is served from `experiments.csv` and its checkpoint.

* **Headline validation metric:** mean ± SD of the 5 per-fold AUCs, plus the **pooled out-of-fold (OOF) AUC** with a
  patient-level bootstrap CI. Fold 0 was also the selection fold, so its AUC is optimistically biased; the mean over
  folds 1–4 is printed as a check.
* **Threshold** (Youden's J) and **temperature scaling** are fitted on OOF predictions only, then frozen for the test
  run (PRD §9).
* Calibration (reliability diagram, Brier score), ROC with a bootstrap CI band, PR curve, confusion matrix, and
  subgroup AUCs (abnormality type, view, density, BI-RADS assessment, subtlety), each with n, CI and a small-group flag.
* Every number goes to `results/metrics.json` by code.

In [ ]:
# ---------------- shared evaluation helpers (used by milestones 13, 14 and 16)
from scipy.optimize import minimize_scalar
from sklearn.metrics import precision_recall_curve, brier_score_loss, confusion_matrix

METRICS_PATH = cfg.results_dir / "metrics.json"


def M(value, split, ci=None, **extra):
    '''One metrics.json entry. value must come from code, never typed by hand.'''
    return {"value": None if value is None else float(value), "split": split,
            "ci95": None if ci is None else [float(ci[0]), float(ci[1])], **extra}


def record_metrics(entries=None, section=None, payload=None):
    '''Merge entries into results/metrics.json (atomic write).'''
    m = json.loads(METRICS_PATH.read_text()) if METRICS_PATH.exists() else {"metrics": {}}
    m.setdefault("metrics", {}).update(entries or {})
    if section:
        m[section] = payload
    m["updated_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    tmp = METRICS_PATH.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(m, indent=2, default=float))
    os.replace(tmp, METRICS_PATH)
    return m


def to_logit(p):
    p = np.clip(np.asarray(p, dtype=np.float64), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def fit_temperature(y, p):
    '''Single temperature T on logits, minimising NLL (fitted on validation predictions only).'''
    z, y = to_logit(p), np.asarray(y, dtype=np.float64)
    nll = lambda T: float(np.mean(np.logaddexp(0, z / T) - y * z / T))
    return float(minimize_scalar(nll, bounds=(0.05, 20.0), method="bounded").x)


def apply_temperature(p, T):
    return 1 / (1 + np.exp(-to_logit(p) / T))


def youden_threshold(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    j = tpr - fpr
    return float(thr[int(np.argmax(j))])


def at_threshold(y, p, thr):
    y, yhat = np.asarray(y), (np.asarray(p) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    return {"threshold": thr, "sensitivity": tp / max(tp + fn, 1), "specificity": tn / max(tn + fp, 1),
            "ppv": tp / max(tp + fp, 1), "npv": tn / max(tn + fn, 1), "tp": int(tp), "fp": int(fp), "tn": int(tn),
            "fn": int(fn)}


def roc_band(y, s, groups, n_boot=1000, seed=0):
    '''Patient-level bootstrap band for the ROC curve (TPR percentiles on a fixed FPR grid).'''
    y, s, groups = np.asarray(y), np.asarray(s), np.asarray(groups)
    uniq, inv = np.unique(groups, return_inverse=True)
    idx_by_g = [np.flatnonzero(inv == g) for g in range(len(uniq))]
    grid, rng, tprs = np.linspace(0, 1, 101), np.random.default_rng(seed), []
    for _ in range(n_boot):
        ii = np.concatenate([idx_by_g[g] for g in rng.integers(0, len(uniq), len(uniq))])
        if len(np.unique(y[ii])) < 2:
            continue
        fpr, tpr, _ = roc_curve(y[ii], s[ii])
        tprs.append(np.interp(grid, fpr, tpr))
    tprs = np.array(tprs)
    return grid, np.percentile(tprs, 2.5, axis=0), np.percentile(tprs, 97.5, axis=0)


SUBGROUP_COLS = {"abn_type": "abnormality type", "view": "view", "breast_density": "breast density",
                 "assessment_max": "BI-RADS assessment (max)", "subtlety_max": "subtlety (max)"}


def subgroup_table(df, split, min_n=50, min_pos=10, n_boot=1000, seed=0):
    rows = []
    for col, nice in SUBGROUP_COLS.items():
        if col not in df.columns:
            continue
        for val, g in df.groupby(df[col].astype(str), sort=True):
            n, n_pos = len(g), int(g.label.sum())
            if g.label.nunique() < 2:
                auc, lo, hi = None, None, None
            else:
                auc = float(roc_auc_score(g.label, g.prob))
                lo, hi, _ = bootstrap_auc(g.label.values, g.prob.values, g.base_pid.values, n_boot, seed)
            rows.append({"split": split, "subgroup": nice, "value": val, "n": n, "n_pos": n_pos, "auc": auc,
                         "ci_lo": lo, "ci_hi": hi,
                         "flag": "one class — no AUC" if auc is None else ("small" if n < min_n or n_pos < min_pos else "")})
    return pd.DataFrame(rows)


def eval_figures(df, tag, thr, title):
    '''ROC (+ bootstrap band), PR, confusion matrix at `thr`, reliability diagram. df needs label, prob, base_pid.'''
    y, p, g = df.label.values, df.prob.values, df.base_pid.values
    fig, ax = plt.subplots(1, 4, figsize=(17, 4))
    fpr, tpr, _ = roc_curve(y, p)
    grid, lo, hi = roc_band(y, p, g)
    ax[0].fill_between(grid, lo, hi, alpha=0.25, label="95% CI (patient bootstrap)")
    ax[0].plot(fpr, tpr, lw=1.5, label=f"AUC {roc_auc_score(y, p):.3f}")
    ax[0].plot([0, 1], [0, 1], "k:", lw=0.8)
    ax[0].set(xlabel="1 − specificity", ylabel="sensitivity", title="ROC"); ax[0].legend(fontsize=7, loc="lower right")
    prec, rec, _ = precision_recall_curve(y, p)
    ax[1].plot(rec, prec, lw=1.5, label=f"PR-AUC {average_precision_score(y, p):.3f}")
    ax[1].axhline(y.mean(), color="k", ls=":", lw=0.8, label=f"prevalence {y.mean():.2f}")
    ax[1].set(xlabel="recall", ylabel="precision", title="Precision–recall"); ax[1].legend(fontsize=7)
    cm = confusion_matrix(y, (p >= thr).astype(int), labels=[0, 1])
    ax[2].imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax[2].text(j, i, str(v), ha="center", va="center", fontsize=11)
    ax[2].set(xticks=[0, 1], yticks=[0, 1], xticklabels=["benign", "malignant"], yticklabels=["benign", "malignant"],
              xlabel="predicted", ylabel="true", title=f"Threshold {thr:.3f} (from OOF)")
    bins = np.linspace(0, 1, 11)
    idx = np.clip(np.digitize(p, bins) - 1, 0, 9)
    centers = [p[idx == b].mean() for b in range(10) if (idx == b).any()]
    fracs = [y[idx == b].mean() for b in range(10) if (idx == b).any()]
    ax[3].plot([0, 1], [0, 1], "k:", lw=0.8)
    ax[3].plot(centers, fracs, "o-", lw=1.5, label=f"Brier {brier_score_loss(y, p):.3f}")
    ax[3].set(xlabel="mean predicted probability", ylabel="observed malignant fraction", title="Reliability")
    ax[3].legend(fontsize=7)
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    save_fig(fig, f"{tag}_evaluation.png")
    plt.show()
    plt.close(fig)

In [ ]:
# ---- 13.1 Train the chosen configuration on all folds (behind RUN_CV)
FINAL = replace(BEST, name=f"s4_cv_{BEST.name}", stage="4", parent=BEST.name,
                hypothesis="Chosen configuration on all 5 folds; headline validation metric (PRD §6).")
EXP_REGISTRY[FINAL.name] = FINAL
print("Final configuration:", json.dumps({k: v for k, v in dataclasses.asdict(FINAL).items()
                                         if k not in ("hypothesis",)}, default=str))
CV_ROWS = {}
OOF_PATH = cfg.results_dir / "oof_predictions.csv"
if cfg.RUN_CV:
    for f in range(cfg.n_folds):
        CV_ROWS[f] = run_experiment(FINAL, fold=f)
else:
    # Use whatever folds already finished (nothing is trained here).
    _ex = load_experiments()
    for f in range(cfg.n_folds):
        hit = _ex[(_ex.exp_hash == FINAL.exp_hash) & (_ex.fold == f) & (_ex.status == "done")] if len(_ex) else _ex
        if len(hit):
            CV_ROWS[f] = hit.iloc[-1].to_dict()
    print(f"ℹ️  RUN_CV is False — {len(CV_ROWS)}/{cfg.n_folds} folds of {FINAL.name} found in experiments.csv; "
          "set cfg.RUN_CV = True to train the rest.")
CV_COMPLETE = len(CV_ROWS) == cfg.n_folds

In [ ]:
# ---- 13.2 Out-of-fold predictions, headline metric, threshold and calibration
CV_RESULT = None
if not CV_COMPLETE:
    print(f"⏸  Cross-validation incomplete ({len(CV_ROWS)}/{cfg.n_folds} folds) — milestone 13 metrics are not computed.")
else:
    _parts = []
    for f, row in CV_ROWS.items():
        pf = cfg.results_dir / "preds" / f"val_{row['exp']}_fold{f}.csv"
        assert pf.exists(), f"missing validation predictions for fold {f}: {pf}"
        _parts.append(pd.read_csv(pf).assign(fold=f))
    OOF = pd.concat(_parts, ignore_index=True)
    _extra = [c for c in ["breast_density", "assessment_max", "subtlety_max", "pathologies", "png_path"] if c in DEV_M.columns]
    OOF = OOF.drop(columns=[c for c in _extra if c in OOF.columns]).merge(DEV_M[["image_key"] + _extra], on="image_key",
                                                                         how="left", validate="1:1")
    # OOF must cover every dev image exactly once, each predicted by the model that did not train on it.
    assert OOF.image_key.is_unique and set(OOF.image_key) == set(DEV_M.image_key), "OOF does not cover the dev set exactly once"
    assert (OOF.set_index("image_key").fold == DEV_M.set_index("image_key").fold.reindex(OOF.image_key)).all(), \
        "an OOF prediction came from the wrong fold"
    assert (OOF.set_index("image_key").label == DEV_M.set_index("image_key").label.reindex(OOF.image_key)).all()

    per_fold = pd.DataFrame([{"fold": f, "val_auc": r["val_auc"], "ci_lo": r["auc_ci_lo"], "ci_hi": r["auc_ci_hi"],
                              "n": int(r["n_val"]), "best_epoch": int(r["best_epoch"]), "gpu_minutes": r["gpu_minutes"],
                              "exp": r["exp"]} for f, r in sorted(CV_ROWS.items())])
    pooled = binary_metrics(OOF.label.values, OOF.prob.values, OOF.base_pid.values, seed=cfg.seed)
    mean_auc, sd_auc = float(per_fold.val_auc.mean()), float(per_fold.val_auc.std(ddof=1))
    mean_1_4 = float(per_fold.loc[per_fold.fold != cfg.fold, "val_auc"].mean())

    T_CAL = fit_temperature(OOF.label.values, OOF.prob.values)
    OOF["prob_cal"] = apply_temperature(OOF.prob.values, T_CAL)
    THRESHOLD = youden_threshold(OOF.label.values, OOF.prob_cal.values)
    oof_thr = at_threshold(OOF.label.values, OOF.prob_cal.values, THRESHOLD)
    brier_raw = float(brier_score_loss(OOF.label, OOF.prob))
    brier_cal = float(brier_score_loss(OOF.label, OOF.prob_cal))
    OOF.to_csv(OOF_PATH, index=False)  # prob = raw model output, prob_cal = after OOF temperature scaling

    print(per_fold.round(4).to_string(index=False))
    print(f"\nHeadline (5-fold patient-grouped CV in the official train split):\n"
          f"  mean ± SD of per-fold AUC : {mean_auc:.4f} ± {sd_auc:.4f}   (folds ≠ {cfg.fold}, not used for selection: "
          f"{mean_1_4:.4f})\n"
          f"  pooled OOF AUC            : {pooled['auc']:.4f} [{pooled['auc_ci95'][0]:.4f}, {pooled['auc_ci95'][1]:.4f}] "
          f"(n={pooled['n']}, {pooled['n_patients']} patients)\n"
          f"  target 0.90               : {'met' if mean_auc >= 0.90 else f'missed by {0.90 - mean_auc:.4f} (mean per-fold AUC)'}")
    print(f"Temperature T = {T_CAL:.3f} | Brier raw {brier_raw:.4f} → calibrated {brier_cal:.4f} | "
          f"threshold (Youden, calibrated OOF) {THRESHOLD:.4f}: sens {oof_thr['sensitivity']:.3f}, spec {oof_thr['specificity']:.3f}")

    _split = "5-fold patient-grouped CV in official train (out-of-fold)"
    _oof_eval = OOF.assign(prob=OOF.prob_cal)
    eval_figures(_oof_eval, "cv_oof", THRESHOLD, f"{FINAL.name} — out-of-fold validation (n={len(OOF)})")
    SUB_OOF = subgroup_table(_oof_eval, "OOF validation")
    SUB_OOF.to_csv(cfg.results_dir / "subgroups_oof.csv", index=False)
    print(SUB_OOF.round(4).to_string(index=False))

    # Training curves of every fold in one figure (from the stored histories)
    _hist = {}
    for f in range(cfg.n_folds):
        _dp = run_dir(FINAL, f) / "done.json"
        if _dp.exists():
            _hist[f] = pd.DataFrame(json.loads(_dp.read_text())["history"])
    if _hist:
        fig, ax = plt.subplots(1, 2, figsize=(11, 3))
        for f, h in _hist.items():
            ax[0].plot(h.epoch, h.val_loss, label=f"fold {f}"); ax[1].plot(h.epoch, h.val_auc, label=f"fold {f}")
        ax[0].set(title="validation loss", xlabel="epoch"); ax[1].set(title="validation AUC", xlabel="epoch")
        ax[1].legend(fontsize=7); fig.tight_layout(); save_fig(fig, "cv_training_curves.png"); plt.show(); plt.close(fig)

    CV_RESULT = {"exp": FINAL.name, "train_hash": FINAL.train_hash, "exp_hash": FINAL.exp_hash,
                 "config": dataclasses.asdict(FINAL), "per_fold": per_fold.to_dict(orient="records"),
                 "mean_auc": mean_auc, "sd_auc": sd_auc, "mean_auc_excluding_selection_fold": mean_1_4,
                 "pooled": pooled, "temperature": T_CAL, "threshold": THRESHOLD, "threshold_rule": "Youden J on calibrated OOF",
                 "oof_at_threshold": oof_thr, "brier_raw": brier_raw, "brier_calibrated": brier_cal,
                 "selection_fold": cfg.fold, "n_configurations_evaluated": int(load_experiments().exp_hash.nunique())}
    record_metrics({
        "cv_auc_mean": M(mean_auc, _split, sd=sd_auc, n_folds=cfg.n_folds),
        "cv_auc_mean_excluding_selection_fold": M(mean_1_4, _split + f", folds ≠ {cfg.fold}"),
        "cv_auc_pooled_oof": M(pooled["auc"], _split, pooled["auc_ci95"], n=pooled["n"], n_pos=pooled["n_pos"]),
        "cv_pr_auc_pooled_oof": M(pooled["pr_auc"], _split),
        "cv_sens_at_spec80": M(pooled["sens_at_spec80"], _split),
        "cv_sens_at_spec90": M(pooled["sens_at_spec90"], _split),
        "cv_spec_at_sens90": M(pooled["spec_at_sens90"], _split),
        "cv_brier_calibrated": M(brier_cal, _split),
    }, section="cross_validation", payload=CV_RESULT)
    print(f"✅ metrics.json updated ({METRICS_PATH})")

### Review gate 3 — before the test run

Review before anything touches the official test split:

* the 5-fold headline (mean ± SD, pooled OOF AUC) against the 0.90 target, and the gap if it is missed;
* the number of configurations tried (`experiments.csv`) and the fold-0 selection bias;
* the frozen threshold and temperature (fitted on OOF only);
* subgroup results and calibration.

Then set `GATE3_PRE_TEST_APPROVED = True` **and**, only with explicit approval in chat, `FINAL_EVAL = True`, and re-run.
With `FINAL_EVAL = True`, milestone 4 downloads the test DICOMs and milestone 14 evaluates them **once**.

In [ ]:
_n_cfg = int(load_experiments().exp_hash.nunique()) if len(load_experiments()) else 0
review_gate("Review gate 3 (before the official test run)", cfg.GATE3_PRE_TEST_APPROVED,
            (f"5-fold CV {FINAL.name}: mean AUC {CV_RESULT['mean_auc']:.4f} ± {CV_RESULT['sd_auc']:.4f}, pooled OOF "
             f"{CV_RESULT['pooled']['auc']:.4f} [{CV_RESULT['pooled']['auc_ci95'][0]:.4f}, "
             f"{CV_RESULT['pooled']['auc_ci95'][1]:.4f}]; threshold {CV_RESULT['threshold']:.4f}, T {CV_RESULT['temperature']:.3f}; "
             f"{_n_cfg} configurations evaluated.") if CV_RESULT else
            f"Cross-validation is incomplete ({len(CV_ROWS)}/{cfg.n_folds} folds) — finish milestone 13 first.")
assert CV_RESULT is not None, "Gate 3 approved but cross-validation is incomplete — finish milestone 13 first."

## 14. Final test evaluation (run once, behind `FINAL_EVAL`)

* **Frozen model first:** the SHA-256 of every fold's `best.pt`, the configuration, threshold and temperature are
  written to `results/frozen_model.json` **before** any test pixel is read; each checkpoint is re-hashed before use.
* Test images get exactly the same preprocessing as train (milestone 7 function), cached on Drive under `test`.
* **Prediction = average of the 5 fold models** (ensemble; hflip TTA if the configuration uses it), then the OOF
  temperature; the OOF threshold is applied unchanged. Single-model test AUCs (each fold model alone) are reported
  separately.
* Nothing here feeds back into training, selection, threshold or calibration. Re-running re-scores the saved test
  predictions; it never re-predicts with a different model.

In [ ]:
TEST_PRED_PATH = cfg.results_dir / "test_predictions.csv"
FROZEN_PATH = cfg.results_dir / "frozen_model.json"
TEST_RESULT = None


def freeze_final_model():
    '''Record the hashes of the 5 fold checkpoints + config/threshold/temperature; idempotent and checked on re-run.'''
    folds = {}
    for f in range(cfg.n_folds):
        bp = run_dir(FINAL, f) / "best.pt"
        assert bp.exists(), f"missing checkpoint for fold {f}: {bp}"
        folds[str(f)] = {"path": str(bp), "sha256": file_sha256(bp)}
    frozen = {"exp": FINAL.name, "train_hash": FINAL.train_hash, "config": dataclasses.asdict(FINAL), "folds": folds,
              "threshold": CV_RESULT["threshold"], "temperature": CV_RESULT["temperature"],
              "manifest_sha256": MANIFEST_SHA256, "folds_sha256": FOLDS_SHA256, "git_commit": git_commit()}
    if FROZEN_PATH.exists():
        old = json.loads(FROZEN_PATH.read_text())
        for k in ("train_hash", "threshold", "temperature"):
            assert old[k] == frozen[k], (f"frozen model changed ({k}): {old[k]} → {frozen[k]}. If the official test split "
                                         "has NOT been evaluated yet, delete results/frozen_model.json to re-freeze.")
        assert {f: v["sha256"] for f, v in old["folds"].items()} == {f: v["sha256"] for f, v in folds.items()}, \
            "a fold checkpoint changed after the model was frozen"
        return old
    frozen["frozen_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    FROZEN_PATH.write_text(json.dumps(frozen, indent=2, default=str))
    return frozen


def build_or_restore_cache(split, images):
    '''Same preprocessing as the train cache (milestone 7), for another split. Returns its manifest.'''
    if cache_complete(split):
        return restore_cache(split)
    assert images.dicom_path.notna().all(), f"{split} DICOMs are missing (milestone 4 downloads them only when FINAL_EVAL)"
    out_dir = cfg.png_dir / split
    out_dir.mkdir(parents=True, exist_ok=True)
    jobs = [{"image_key": r.image_key, "dicom_path": r.dicom_path, "params": PARAMS, "out_path": str(out_dir / f"{r.image_key}.png"),
             "out_rel": f"{split}/{r.image_key}.png", "out_uncropped": None, "out_uncropped_rel": None}
            for r in images.itertuples()]
    print(f"Preprocessing {len(jobs):,} {split} images …", flush=True)
    ctx = mp.get_context("fork") if "fork" in mp.get_all_start_methods() else None
    recs = []
    with ProcessPoolExecutor(max_workers=cfg.n_workers, mp_context=ctx) as ex:
        for i, fut in enumerate(as_completed([ex.submit(preprocess_one, j) for j in jobs]), 1):
            recs.append(fut.result())
            if i % 100 == 0 or i == len(jobs):
                print(f"  {i}/{len(jobs)}", flush=True)
    recs = pd.DataFrame(recs)
    errs = recs[recs.error.notna()]
    assert errs.empty, f"{len(errs)} {split} images failed preprocessing:\n{errs.head(20).to_string()}"
    keep = ["image_key", "split", "base_pid", "side", "view", "label", "abn_type", "n_abnormalities", "pathologies",
            "breast_density", "assessment_max", "subtlety_max", "series_uid"]
    man = images[keep].merge(recs.drop(columns=["error"]), on="image_key", how="left", validate="1:1")
    mpath = cfg.manifest_path(split)
    man.to_csv(mpath, index=False)
    tar_tmp = cfg.cache_tar(split).with_suffix(".tar.part")
    sh(f"tar -cf '{tar_tmp}' -C '{cfg.png_dir}' {split}", check=True)
    tar_tmp.rename(cfg.cache_tar(split))
    cfg.cache_marker(split).write_text(json.dumps({
        "split": split, "cache_version": cfg.cache_version, "n_images": int(len(man)),
        "manifest_sha256": hashlib.sha256(mpath.read_bytes()).hexdigest(), "params": PARAMS,
        "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}, indent=2, default=str))
    return man


def load_fold_model(f, frozen):
    bp = Path(frozen["folds"][str(f)]["path"])
    assert file_sha256(bp) == frozen["folds"][str(f)]["sha256"], f"fold {f} checkpoint hash mismatch"
    m = build_model(FINAL, pretrained=False)
    m.load_state_dict(torch.load(bp, map_location="cpu", weights_only=False)["model"])
    return m.eval()


def run_final_eval(test_df, split_name):
    '''Ensemble prediction on `test_df` (needs image_key, base_pid, label, png_path + subgroup columns); idempotent.'''
    frozen = freeze_final_model()
    if TEST_PRED_PATH.exists():
        preds = pd.read_csv(TEST_PRED_PATH)
        assert set(preds.image_key) == set(test_df.image_key), "saved test predictions do not match the test set"
        print(f"↺ re-scoring saved test predictions ({TEST_PRED_PATH}); no new inference.")
    else:
        bs = max(1, probe_batch_size(FINAL))
        preds = test_df[["image_key", "base_pid", "label"]].copy()
        for f in range(cfg.n_folds):
            model = load_fold_model(f, frozen)
            preds[f"prob_fold{f}"] = predict(model, make_loader(test_df, FINAL.image_size, False, bs * 2),
                                             tta_hflip=FINAL.tta_hflip)
            del model
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            print(f"  fold {f} model: done", flush=True)
        preds["prob_raw"] = preds[[f"prob_fold{f}" for f in range(cfg.n_folds)]].mean(axis=1)
        preds["prob"] = apply_temperature(preds.prob_raw.values, frozen["temperature"])
        preds.to_csv(TEST_PRED_PATH, index=False)
    sub_cols = [c for c in list(SUBGROUP_COLS) + ["pathologies", "png_path", "side"] if c in test_df.columns and c not in preds.columns]
    df = preds.merge(test_df[["image_key"] + sub_cols], on="image_key", how="left", validate="1:1")
    y, g = df.label.values, df.base_pid.values
    ens = binary_metrics(y, df.prob.values, g, seed=cfg.seed)
    singles = [float(roc_auc_score(y, df[f"prob_fold{f}"])) for f in range(cfg.n_folds)]
    thr = at_threshold(y, df.prob.values, frozen["threshold"])
    brier = float(brier_score_loss(y, df.prob))
    print(f"{split_name}: ensemble AUC {ens['auc']:.4f} [{ens['auc_ci95'][0]:.4f}, {ens['auc_ci95'][1]:.4f}] "
          f"(n={ens['n']}, {ens['n_pos']} malignant, {ens['n_patients']} patients) | single models "
          f"{np.mean(singles):.4f} ± {np.std(singles, ddof=1):.4f} | at OOF threshold {frozen['threshold']:.4f}: "
          f"sens {thr['sensitivity']:.3f}, spec {thr['specificity']:.3f} | Brier {brier:.4f}")
    eval_figures(df, "test", frozen["threshold"], f"{FINAL.name} — {split_name} (5-model ensemble, n={len(df)})")
    sub = subgroup_table(df, split_name)
    print(sub.round(4).to_string(index=False))
    return {"df": df, "ensemble": ens, "single_model_aucs": singles, "at_threshold": thr, "brier": brier, "subgroups": sub}


if not cfg.FINAL_EVAL:
    print("🔒 FINAL_EVAL is False — the official test split is not loaded (PRD §6). Approve gate 3, then set FINAL_EVAL "
          "only with explicit approval.")
else:
    assert cfg.GATE3_PRE_TEST_APPROVED, "FINAL_EVAL requires review gate 3 approval"
    freeze_final_model()  # hashes recorded before any test pixel is read
    TEST_IMAGES = IMAGES[IMAGES.split == "test"].reset_index(drop=True)
    TEST_MANIFEST = build_or_restore_cache("test", TEST_IMAGES)
    _test_df = TEST_MANIFEST.copy()
    assert len(_test_df) == len(TEST_IMAGES) and set(_test_df.image_key) == set(TEST_IMAGES.image_key)
    _split = f"official test ({len(_test_df)} images, 5-model ensemble)"
    TEST_RESULT = run_final_eval(_test_df, _split)
    TEST_RESULT["subgroups"].to_csv(cfg.results_dir / "subgroups_test.csv", index=False)
    e = TEST_RESULT["ensemble"]
    record_metrics({
        "test_auc_ensemble": M(e["auc"], _split, e["auc_ci95"], n=e["n"], n_pos=e["n_pos"]),
        "test_auc_single_model_mean": M(np.mean(TEST_RESULT["single_model_aucs"]), "official test, each fold model alone",
                                        sd=float(np.std(TEST_RESULT["single_model_aucs"], ddof=1))),
        "test_pr_auc_ensemble": M(e["pr_auc"], _split),
        "test_sens_at_oof_threshold": M(TEST_RESULT["at_threshold"]["sensitivity"], _split),
        "test_spec_at_oof_threshold": M(TEST_RESULT["at_threshold"]["specificity"], _split),
        "test_sens_at_spec90": M(e["sens_at_spec90"], _split),
        "test_brier": M(TEST_RESULT["brier"], _split),
    }, section="test", payload={k: v for k, v in TEST_RESULT.items() if k not in ("df", "subgroups")})
    print(f"✅ metrics.json updated ({METRICS_PATH})")

## 15. Explainability — Grad-CAM

Grad-CAM on the backbone's last feature map, for true positives, false positives and false negatives. Validation
images are explained by the fold model that **did not** train on them; test images (only after milestone 14) by the
ensemble's fold-0 model. The ground-truth ROI outline (CBIS-DDSM masks; only the few series shown are downloaded) is
drawn for comparison. DDSM outlines are imprecise, so the overlays are illustrations, not a localisation metric.

In [ ]:
def gradcam(model, x):
    '''x: (1,1,H,W) tensor. Returns (cam HxW in [0,1], probability).'''
    model.eval()
    x = x.to(DEVICE).float().requires_grad_(False)
    feats = model.backbone.forward_features(x)
    feats.retain_grad()
    logit = model.head(model.drop(model.pool(feats))).squeeze()
    model.zero_grad(set_to_none=True)
    logit.backward()
    w = feats.grad.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * feats).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0].detach().float().cpu().numpy()
    cam = cam / cam.max() if cam.max() > 0 else cam
    return cam, float(torch.sigmoid(logit).item())


def roi_outline(image_key, split, image_size, man_row):
    '''Union ROI mask for one image in training coordinates, or None. Downloads only this image's ROI series.'''
    if IDC is None:
        return None
    rows = ABN[(ABN.split == split) & ((split + "_" + ABN.base_pid + "_" + ABN.side + "_" + ABN.view) == image_key)]
    union = None
    for r in rows.itertuples():
        cands = []
        for p in (r.roi_mask_file_path, r.cropped_image_file_path):
            u = parse_dicom_path(p)[2]
            if u and not series_on_disk(u) and u in set(IDC.SeriesInstanceUID):
                idc_client.download_from_selection(downloadDir=str(cfg.dicom_dir), seriesInstanceUID=[u],
                                                   dirTemplate="%SeriesInstanceUID", quiet=True, show_progress_bar=False)
            cands += [(pydicom.dcmread(f, stop_before_pixels=True), f) for f in series_on_disk(u)] if u else []
        exact = [f for ds, f in cands if (int(ds.Rows), int(ds.Columns)) == (int(man_row.orig_h), int(man_row.orig_w))]
        if not exact:
            continue
        m = mask_to_train_coords(pydicom.dcmread(exact[0]).pixel_array, man_row, image_size)
        union = m if union is None else (union | m)
    return union


def gradcam_panel(df, split, models_by_fold, tag, n=3):
    '''df: predictions with label, prob(_cal), fold (or None → fold 0 model), png_path. Saves one figure.'''
    thr = CV_RESULT["threshold"]
    groups = {"true positive": df[(df.label == 1) & (df.prob >= thr)].sort_values("prob", ascending=False),
              "false positive": df[(df.label == 0) & (df.prob >= thr)].sort_values("prob", ascending=False),
              "false negative": df[(df.label == 1) & (df.prob < thr)].sort_values("prob")}
    fig, axes = plt.subplots(len(groups), 2 * n, figsize=(2.2 * 2 * n, 3.3 * len(groups)), squeeze=False)
    for gi, (name, g) in enumerate(groups.items()):
        for j in range(n):
            a0, a1 = axes[gi, 2 * j], axes[gi, 2 * j + 1]
            a0.axis("off"); a1.axis("off")
            if j >= len(g):
                continue
            r = g.iloc[j]
            f = int(r["fold"]) if "fold" in g.columns and not pd.isna(r.get("fold")) else 0
            ds = MammoDataset(g.iloc[[j]], FINAL.image_size, train=False)
            x, _, _ = ds[0]
            cam, p = gradcam(models_by_fold[f], x[None])
            img = x[0].numpy()
            a0.imshow(img, cmap="gray", vmin=-2.5, vmax=2.5)
            a1.imshow(img, cmap="gray", vmin=-2.5, vmax=2.5); a1.imshow(cam, cmap="jet", alpha=0.35)
            roi = roi_outline(r["image_key"], split, FINAL.image_size, r)
            if roi is not None and roi.any():
                for a in (a0, a1):
                    a.contour(roi, levels=[0.5], colors="lime", linewidths=0.8)
            a0.set_title(f"{name}\n{r['image_key'].split('_', 1)[1]}", fontsize=6)
            a1.set_title(f"p={r['prob']:.2f} {r.get('abn_type', '')}", fontsize=6)
    fig.tight_layout()
    save_fig(fig, f"gradcam_{tag}.png")
    plt.show()
    plt.close(fig)


if CV_RESULT is None:
    print("⏸  Grad-CAM skipped: cross-validation incomplete.")
elif not cfg.RUN_GRADCAM:
    print("ℹ️  Grad-CAM skipped (cfg.RUN_GRADCAM is False).")
else:
    _frozen = freeze_final_model()
    _models = {f: load_fold_model(f, _frozen) for f in range(cfg.n_folds)}
    _oof = pd.read_csv(OOF_PATH).drop(columns=["prob"]).rename(columns={"prob_cal": "prob"})
    _oof = _oof.merge(MANIFEST[["image_key", "orig_h", "orig_w", "crop_y0", "crop_y1", "crop_x0", "crop_x1", "flipped",
                                "out_h", "out_w"]], on="image_key", how="left")
    gradcam_panel(_oof, "train", _models, "oof_validation")
    if TEST_RESULT is not None:
        _tdf = TEST_RESULT["df"].merge(TEST_MANIFEST[["image_key", "orig_h", "orig_w", "crop_y0", "crop_y1", "crop_x0",
                                                      "crop_x1", "flipped", "out_h", "out_w"]], on="image_key", how="left")
        gradcam_panel(_tdf.drop(columns=[c for c in ["fold"] if c in _tdf.columns]), "test", _models, "test")
    del _models
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

## 16. Error analysis

The most confident false positives and false negatives (at the frozen OOF threshold), with their metadata and
thumbnails, plus how errors split across subgroups. Validation (OOF) always; test only after milestone 14. The counts
printed here are what the model card's failure modes are based on.

In [ ]:
def error_analysis(df, tag, k=None):
    k = k or cfg.n_error_examples
    thr = CV_RESULT["threshold"]
    df = df.assign(pred=(df.prob >= thr).astype(int))
    fp = df[(df.label == 0) & (df.pred == 1)].sort_values("prob", ascending=False)
    fn = df[(df.label == 1) & (df.pred == 0)].sort_values("prob")
    cols = [c for c in ["image_key", "prob", "abn_type", "view", "breast_density", "assessment_max", "subtlety_max",
                        "pathologies"] if c in df.columns]
    print(f"[{tag}] threshold {thr:.4f}: {len(fp)} false positives, {len(fn)} false negatives of {len(df)} images")
    print(f"\nMost confident false positives (benign called malignant):\n{fp[cols].head(k).round(3).to_string(index=False)}")
    print(f"\nMost confident false negatives (malignant called benign):\n{fn[cols].head(k).round(3).to_string(index=False)}")
    rows = []
    for col in ["abn_type", "view", "breast_density", "assessment_max", "subtlety_max"]:
        if col not in df.columns:
            continue
        for val, g in df.groupby(df[col].astype(str)):
            pos, neg = g[g.label == 1], g[g.label == 0]
            rows.append({"subgroup": col, "value": val, "n": len(g), "malignant": len(pos),
                         "fn_rate": (pos.pred == 0).mean() if len(pos) else np.nan,
                         "fp_rate": (neg.pred == 1).mean() if len(neg) else np.nan})
    rates = pd.DataFrame(rows)
    rates.to_csv(cfg.results_dir / f"error_rates_{tag}.csv", index=False)
    print(f"\nError rates by subgroup (FN rate among malignant, FP rate among benign):\n{rates.round(3).to_string(index=False)}")
    fig, axes = plt.subplots(2, k, figsize=(1.7 * k, 6), squeeze=False)
    for gi, (name, g) in enumerate([("FP", fp), ("FN", fn)]):
        for j in range(k):
            ax = axes[gi, j]; ax.axis("off")
            if j < len(g):
                r = g.iloc[j]
                im = cv2.imread(str(cfg.png_dir / r["png_path"]), cv2.IMREAD_UNCHANGED)
                ax.imshow(to_uint8_preview(im, 320), cmap="gray")
                ax.set_title(f"{name} p={r['prob']:.2f}\n{r.get('abn_type', '')} {r.get('view', '')}", fontsize=6)
    fig.tight_layout(); save_fig(fig, f"errors_{tag}.png"); plt.show(); plt.close(fig)
    return {"n_fp": int(len(fp)), "n_fn": int(len(fn)), "n": int(len(df)), "rates": rates.to_dict(orient="records")}


ERRORS = {}
if CV_RESULT is None:
    print("⏸  Error analysis skipped: cross-validation incomplete.")
else:
    _oof = pd.read_csv(OOF_PATH).drop(columns=["prob"]).rename(columns={"prob_cal": "prob"})
    ERRORS["oof"] = error_analysis(_oof, "oof")
    if TEST_RESULT is not None:
        ERRORS["test"] = error_analysis(TEST_RESULT["df"], "test")
    record_metrics(section="error_analysis", payload=ERRORS)

## 17. Export

* `results/metrics.json` (every reported number, written by code above), `experiments.csv`, `oof_predictions.csv`,
  `test_predictions.csv` (after milestone 14), subgroup tables and figures — all on Drive under `results/`.
* **Weights:** the 5 frozen fold checkpoints are exported as fp16 `state_dict`s with the config and a model card to
  `export/`. Optional push to the Hugging Face Hub when `cfg.hf_repo_id` is set (token from Colab Secrets `HF_TOKEN`).
* `requirements.lock.txt`: exact package versions of this runtime (PRD §8), to copy into the repo's `requirements.txt`.
* **README results block:** generated from `metrics.json` into `results/README_results.md`, and written between the
  `<!-- RESULTS:START -->` / `<!-- RESULTS:END -->` markers of `README.md` when the repo is present.

In [ ]:
EXPORT_DIR = cfg.drive / "export"


def results_markdown():
    '''README results block — every number read from metrics.json.'''
    m = json.loads(METRICS_PATH.read_text()) if METRICS_PATH.exists() else {"metrics": {}}
    mm = m.get("metrics", {})
    if not mm:
        return "*No results yet.*"
    fmt = lambda e: "—" if e.get("value") is None else f"{e['value']:.4f}" + (f" ± {e['sd']:.4f}" if e.get("sd") is not None else "")
    ci = lambda e: "—" if not e.get("ci95") else f"[{e['ci95'][0]:.4f}, {e['ci95'][1]:.4f}]"
    lines = ["| Metric | Split | Value | 95% CI |", "|---|---|---|---|"]
    for k, e in mm.items():
        lines.append(f"| `{k}` | {e.get('split', '')} | {fmt(e)} | {ci(e)} |")
    cv = m.get("cross_validation") or {}
    if cv:
        lines += ["", f"Configuration: `{cv.get('exp')}` (train hash `{cv.get('train_hash')}`); "
                      f"{cv.get('n_configurations_evaluated')} configurations evaluated on validation (`experiments.csv`). "
                      f"Fold {cv.get('selection_fold')} was the selection fold."]
    lines += ["", f"*Generated from `results/metrics.json` ({m.get('updated_utc', '')}) by notebook milestone 17.*"]
    return "\n".join(lines)


def model_card():
    m = json.loads(METRICS_PATH.read_text()) if METRICS_PATH.exists() else {}
    cv, te, er = m.get("cross_validation") or {}, m.get("test") or {}, m.get("error_analysis") or {}
    lines = [f"# Model card — {FINAL.name}", "",
             "**Intended use:** portfolio and research only. **Not for clinical use**; no clinical claims.", "",
             "**Task:** single-view whole-mammogram benign vs malignant classification on CBIS-DDSM (all images contain a "
             "biopsied lesion; there are no normal images).", "",
             f"**Architecture:** {FINAL.backbone}, input {FINAL.image_size[0]}×{FINAL.image_size[1]}, pooling {FINAL.pool}, "
             f"EMA {FINAL.ema}, TTA hflip {FINAL.tta_hflip}; 5 fold models averaged; temperature scaling and threshold "
             "fitted on out-of-fold validation only.", "",
             "**Data limits:** 1990s US scanned film digitised from analogue mammograms (not FFDM); imprecise DDSM lesion "
             "outlines; label noise; small validation folds.", "",
             "## Results", "", results_markdown(), ""]
    if er:
        lines += ["## Failure modes (from milestone 16, counts by code)", ""]
        for split, e in er.items():
            lines.append(f"- {split}: {e['n_fp']} false positives and {e['n_fn']} false negatives of {e['n']} images at "
                         "the frozen threshold; per-subgroup FN/FP rates in `results/error_rates_*.csv`.")
    lines += ["", "**Citation:** CBIS-DDSM (CC BY 3.0), DOI 10.7937/K9/TCIA.2016.7O02S9CY; Lee et al., Sci Data 4, 170177 "
              "(2017); Clark et al., J Digit Imaging 26 (2013)."]
    return "\n".join(lines)


# requirements lock (exact versions of this runtime)
_lock = [f"{({'sklearn': 'scikit-learn', 'idc_index': 'idc-index'}).get(k, k)}=={v}" for k, v in _versions.items() if v]
(cfg.results_dir / "requirements.lock.txt").write_text(
    f"# Exact versions from the run on {GPU_INFO.get('name')} (Python {platform.python_version()}), written by milestone 17.\n"
    + "\n".join(_lock) + "\n")
print("requirements.lock.txt:", ", ".join(_lock))

(cfg.results_dir / "README_results.md").write_text(results_markdown() + "\n")
for _readme in [Path("README.md"), Path("../README.md")]:
    if _readme.exists() and "<!-- RESULTS:START -->" in _readme.read_text():
        _t = _readme.read_text()
        _a, _rest = _t.split("<!-- RESULTS:START -->", 1)
        _b = _rest.split("<!-- RESULTS:END -->", 1)[1]
        _readme.write_text(_a + "<!-- RESULTS:START -->\n" + results_markdown() + "\n<!-- RESULTS:END -->" + _b)
        print(f"README results block updated: {_readme.resolve()}")
        break
else:
    print(f"README.md with result markers not found here — copy {cfg.results_dir / 'README_results.md'} into the repo.")

if CV_RESULT is not None:
    _frozen = freeze_final_model()
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    for f in range(cfg.n_folds):
        _dst = EXPORT_DIR / f"{FINAL.name}_fold{f}_fp16.pt"
        if not _dst.exists():
            _bp = Path(_frozen["folds"][str(f)]["path"])
            assert file_sha256(_bp) == _frozen["folds"][str(f)]["sha256"]
            _sd = torch.load(_bp, map_location="cpu", weights_only=False)["model"]
            torch.save({k: (v.half() if v.is_floating_point() else v) for k, v in _sd.items()}, _dst)
    (EXPORT_DIR / "config.json").write_text(json.dumps({"exp": dataclasses.asdict(FINAL), "threshold": _frozen["threshold"],
                                                        "temperature": _frozen["temperature"],
                                                        "source_sha256": {f: v["sha256"] for f, v in _frozen["folds"].items()}},
                                                       indent=2, default=str))
    (EXPORT_DIR / "MODEL_CARD.md").write_text(model_card() + "\n")
    print(f"Exported {cfg.n_folds} fp16 fold models + config + model card → {EXPORT_DIR}")
    if cfg.hf_repo_id:
        from google.colab import userdata  # token from Colab Secrets, never from notebook text
        from huggingface_hub import HfApi
        HfApi(token=userdata.get("HF_TOKEN")).upload_folder(folder_path=str(EXPORT_DIR), repo_id=cfg.hf_repo_id,
                                                            repo_type="model")
        print(f"Pushed {EXPORT_DIR} to the Hugging Face Hub: {cfg.hf_repo_id}")
else:
    print("⏸  Weights export skipped: cross-validation incomplete.")

print("\nResults on Drive:")
for p in sorted(cfg.results_dir.glob("*")):
    if p.is_file():
        print(f"  {p.name:40s} {p.stat().st_size / 1e3:9.1f} kB")

## 18. Conclusions and limitations

The verdict below is printed from `metrics.json` — no number in it is typed by hand.

**Limitations (known before any result):**
* **Data:** 1990s US scanned film, digitised from analogue mammograms — results do not transfer to digital (FFDM)
  mammography. No normal images: this is benign vs malignant among biopsied lesions, not screening.
* **Labels:** DDSM lesion outlines are imprecise and pathology labels are per abnormality; an image is malignant if any
  of its abnormalities is. Label noise is certain.
* **Validation:** folds of ~480 images give wide CIs; fold 0 was used to select the configuration, so its AUC is
  optimistic. The headline is the 5-fold mean and pooled OOF AUC.
* **Comparisons:** published numbers come from different splits (PRD §3); only the official-test row is comparable with
  Petrini et al.'s official-split results.
* **Explainability:** Grad-CAM shows where the score is sensitive, not why; it is not a localisation metric.

In [ ]:
_m = json.loads(METRICS_PATH.read_text()) if METRICS_PATH.exists() else {"metrics": {}}
_mm = _m.get("metrics", {})
if "cv_auc_mean" not in _mm:
    print("No cross-validation result yet — conclusions are written after milestone 13 has run.")
else:
    cvm, pooled = _mm["cv_auc_mean"], _mm["cv_auc_pooled_oof"]
    gap = 0.90 - cvm["value"]
    print(f"Validation (5-fold, official train): mean AUC {cvm['value']:.4f} ± {cvm['sd']:.4f}; pooled OOF "
          f"{pooled['value']:.4f} [{pooled['ci95'][0]:.4f}, {pooled['ci95'][1]:.4f}].")
    print(f"Target 0.90: {'MET' if gap <= 0 else f'NOT MET — short by {gap:.4f} (mean per-fold AUC)'}; "
          f"the pooled-OOF CI upper bound is {pooled['ci95'][1]:.4f}.")
    if "test_auc_ensemble" in _mm:
        t = _mm["test_auc_ensemble"]
        print(f"Official test (once, frozen ensemble): AUC {t['value']:.4f} [{t['ci95'][0]:.4f}, {t['ci95'][1]:.4f}] — "
              "compare only with official-split results (Petrini et al. 2025 single view 0.8325 ± 0.0171, PRD §3).")
    else:
        print("Official test: not evaluated yet (FINAL_EVAL is False).")
    _ex = load_experiments()
    print(f"Search size: {_ex.exp_hash.nunique()} configurations on validation, {_ex.gpu_minutes.sum() / 60:.1f} GPU-hours.")